# Deploy the VSS Warehouse Blueprint

This notebook deploys the NVIDIA VSS **warehouse** industry profile on a GPU-equipped cloud
instance (Brev launchable or any Linux GPU host).

The compose files ship **in-tree** in the `video-search-and-summarization` repo — there is no
NGC compose tarball. Only the *app data* (sample videos + perception models) comes from NGC.
On Brev the repo is cloned onto the instance automatically; set `DEPLOY_SOURCE_PATH` in
Section 1 to that path (typically `~/video-search-and-summarization`).

**The deployment is driven by `docker compose` directly.** This notebook does the setup work
itself rather than delegating to a wrapper script:

1. copies `industry-profiles/warehouse-operations/overrides.env` → `generated.env` (Section 8)
2. writes your `MODE` / `BP_PROFILE` / paths / IPs / credentials into `generated.env`
3. selects the `COMPOSE_PROFILES` service list for the chosen variant
   (`COMPOSE_PROFILES_WH_*` — the profile-inversion model: every service carries its own
   compose profile name and each variant selects a flat list)
4. creates and permissions the `$VSS_DATA_DIR/data_log/...` directories (Section 9)
5. runs

   ```bash
   docker compose -f compose.yml -f services/infra/compose-no-turn-tcp-relay.yml \
     --env-file containers.env \
     --env-file industry-profiles/warehouse-operations/.env \
     --env-file industry-profiles/warehouse-operations/generated.env \
     up --detach --force-recreate --build
   ```

> `containers.env` pins the first-party image coordinates, `.env` holds the stable warehouse
> defaults, and `generated.env` is the per-deployment layer (git-ignored, re-created on every
> deploy). The compose files repeat the `containers.env` values as inline `image:` defaults, so
> that one is a knob rather than a hard requirement — but pass all three, as this notebook
> does, or overriding a registry/tag in `containers.env` silently has no effect.

**What this notebook does:**

1. Validates GPU hardware and system prerequisites (Section 2)
2. Installs and configures the NGC CLI (Section 3)
3. Moves Docker/containerd storage to a large volume and pins the Docker version (Section 4)
4. Logs in to `nvcr.io` (Section 5) and downloads the warehouse app data from NGC (Section 6)
5. Detects the network configuration, including Brev secure links (Section 7)
6. Writes the per-deployment `generated.env` (Section 8; `overrides.env` is left untouched)
7. Deploys with `docker compose ... up --detach` (Section 9)
8. Verifies containers, perception FPS and service endpoints (Section 10)
9. Prints the access URLs (Section 11), then stop / teardown (Sections 13–14)

**Deployment variants**

| `MODE` | `BP_PROFILE` | LLM | Notes |
|--------|--------------|-----|-------|
| `2d` | `bp_wh` | `local` / `remote` / `none` | Agents + Agent UI + RT-VLM |
| `2d` | `bp_wh_kafka` / `bp_wh_redis` | none | CV only |
| `3d` | `bp_wh_kafka` / `bp_wh_redis` | none | Sparse4D |
| `mv3dt` | `bp_wh_kafka` / `bp_wh_redis` | none | BEV fusion + MQTT |
| `auto-calibration` | `bp_wh_auto_calib` | none | Single `COMPOSE_PROFILES_WH_AUTO_CALIB` list; direct VST (SDRC commented) |

**Dataset is an independent choice, not a per-mode one.** All three shipped datasets carry a
checked-in `calibration.json` for `2d`, `3d` and `mv3dt`, so any dataset pairs with any mode.
`SAMPLE_VIDEO_DATASET` picks one; `NUM_STREAMS` (4 / 3 / 4) and `DATASET_TYPE`
(`real` / `synthetic`) are derived from it, never set by hand.

`bp_wh` (agents) is **2D-only**. This notebook always deploys the **extended** stack (except auto-calibration, which is already a minimal AMC + nvstreamer + VST subset).

## 1. Configuration

Set your NGC API key and warehouse deployment parameters below. All later cells use these
variables.

**Required:**

- `NGC_CLI_API_KEY` — get one at <https://ngc.nvidia.com>
- `DEPLOY_SOURCE_PATH` — path to the `video-search-and-summarization` checkout. The warehouse
  compose files live in-tree at `<DEPLOY_SOURCE_PATH>/deploy/docker/`.

**Valid `MODE` / `BP_PROFILE` combinations** (enforced in Section 1):

| `MODE` | Valid `BP_PROFILE` |
|--------|--------------------|
| `2d` | `bp_wh` (agents), `bp_wh_kafka`, `bp_wh_redis` |
| `3d` | `bp_wh_kafka`, `bp_wh_redis` |
| `mv3dt` | `bp_wh_kafka`, `bp_wh_redis` |
| `auto-calibration` | `bp_wh_auto_calib` only — one list, `COMPOSE_PROFILES=${COMPOSE_PROFILES_WH_AUTO_CALIB}` |

Auto-calibration uses **direct VST**, not SDRC. Section 8 comments `VST_USE_SDRC`, `STREAM_PROCESSOR_MODULE_ENDPOINT`, and `VST_NGINX_MODE` in `generated.env` so `services/vios/vst.env` defaults apply. Do not comment those keys in the checked-in `overrides.env` (other warehouse variants still need SDRC).

`STREAM_TYPE` is derived, not configured: `redis` for `bp_wh_redis`, `kafka` for everything
else. `NUM_STREAMS` and `DATASET_TYPE` are derived from `SAMPLE_VIDEO_DATASET` — camera count
and footage family are properties of the dataset, not of the mode — so the only dataset knob
is `SAMPLE_VIDEO_DATASET` itself. Leaving it empty falls back to a per-mode default.

**GPU layout** (defaults from `overrides.env`): perception on GPU `0`, RT-VLM on GPU `1`
(`bp_wh` only), LLM NIM on GPU `2` (`bp_wh` + `LLM_MODE=local`). A host with fewer GPUs must
point these at devices that exist — set the `*_DEVICE_ID` values below, or leave
`LLM_MODE="remote"`.

**LLM placement.** Only `MODE=2d` + `BP_PROFILE=bp_wh` runs an LLM at all; every other variant
— including the default `bp_wh_kafka` — is forced to `LLM_MODE=none`, so the whole LLM block is
inert unless you switch to `bp_wh`. When you do, `LLM_MODE` defaults to **`remote`**: no third
GPU, no ~18 GB weight download, and the agent starts as soon as the endpoint answers. It then
needs `REMOTE_LLM_ENDPOINT_URL` plus `NVIDIA_API_KEY` (build.nvidia.com) or `OPENAI_API_KEY`,
and Section 1 refuses to continue without one.
Switch to `local` when you want everything on-box and have a GPU free — that additionally
requires a sizing file at `services/nim/<model-slug>/hw-<HARDWARE_PROFILE>.env`, which the
validation cell checks for you.

In [ ]:
import os as _os

# ============================================================
# REQUIRED: set these before running anything else
# ============================================================

# Your NGC API key — get one at https://ngc.nvidia.com. No default: the deployment
# cannot pull images or download app data without it.
NGC_CLI_API_KEY = ""

# Repo checkout holding deploy/docker. Brev clones it here automatically; change this
# only if your checkout lives elsewhere.
DEPLOY_SOURCE_PATH = _os.path.expanduser("~/video-search-and-summarization")

# ---- Deployment selection ----
# Analytics mode. "2d" = RT-DETR single-view detection; "3d" = depth-aware Sparse4D;
# "mv3dt" = multi-view 3D tracking with BEV fusion; "auto-calibration" produces a
# calibration instead of running analytics. Default "2d" is the lightest to bring up.
MODE = "2d"                   # "2d" | "3d" | "mv3dt" | "auto-calibration"

# Which stack to deploy. Default "bp_wh_kafka" is headless perception plus analytics with
# Kafka as the CV metadata broker — no agent, no UI, no LLM. "bp_wh_redis" is the same
# stack on Redis. "bp_wh" is the only variant carrying an agent, UI and RT-VLM, and it
# also pulls in an LLM (see the LLM block below); it is 2D-only.
# Valid pairings: bp_wh -> 2d only; bp_wh_kafka / bp_wh_redis -> 2d, 3d, mv3dt;
# bp_wh_auto_calib -> auto-calibration only (and vice versa — it is a single stack).
# For auto-calibration, Section 8 writes COMPOSE_PROFILES=${COMPOSE_PROFILES_WH_AUTO_CALIB}
# and comments the SDRC block in generated.env (direct VST).
BP_PROFILE = "bp_wh_kafka"    # "bp_wh" | "bp_wh_kafka" | "bp_wh_redis" | "bp_wh_auto_calib"

# Hardware profile — picks the NIM sizing env files and perception tuning.
# Valid: H100, L4, L40S, RTXA6000, RTXA6000ADA, RTXPRO6000BW, RTXPRO4500BW,
#        IGX-THOR, AGX-THOR, DGX-SPARK, OTHER  (tuning sections live in blueprint_config.yml;
#        a local LLM NIM also needs services/nim/<model>/hw-<PROFILE>.env)
HARDWARE_PROFILE = "RTXPRO6000BW"

# Elasticsearch runtime. "cpu" suits every warehouse variant; "gpu" only pays off with
# GPU-backed vector search, which these profiles do not enable.
ELASTICSEARCH_MODE = "cpu"    # "cpu" | "gpu"

# ---- LLM (MODE=2d + BP_PROFILE=bp_wh only) ----
# Ignored on the default BP_PROFILE="bp_wh_kafka": the next cell forces LLM_MODE="none"
# for every variant except 2d + bp_wh, so nothing below applies unless you switch to it.
# "remote" is the default: it needs no dedicated GPU and skips the ~18 GB NIM weight
# download. Use "local" only when you have a spare GPU AND a matching sizing file at
# services/nim/<model-slug>/hw-<HARDWARE_PROFILE>.env (checked in the next cell).
LLM_MODE = "remote"           # "remote" | "local" | "none"
# Local LLM choices (name -> container/profile slug is derived in Section 1):
#   nvidia/nemotron-3.5-lightning-30b-a3b    nvidia/NVIDIA-Nemotron-Nano-9B-v2-FP8
# Model id. For "remote" this MUST match an id the endpoint advertises, and is sent to the
# endpoint verbatim. Do not leave it to auto-detection: auto-detection would take the FIRST
# id from <endpoint>/v1/models, which on build.nvidia.com is an unrelated model.
#
# The LLM model nvidia/nemotron-3.5-lightning-30b-a3b may no longer be served by build.nvidia.com.
# If the deployment cannot reach the model, replace it with one the endpoint still advertises, e.g.
#     LLM_NAME = "nvidia/nemotron-3-nano-omni-30b-a3b-reasoning"
# Also check the live list at https://integrate.api.nvidia.com/v1/models (or build.nvidia.com) for other options.
LLM_NAME = "nvidia/nemotron-3.5-lightning-30b-a3b"
# Remote LLM (LLM_MODE="remote"): OpenAI-compatible endpoint root, WITHOUT the trailing /v1.
REMOTE_LLM_ENDPOINT_URL = "https://integrate.api.nvidia.com"
LLM_MODEL_TYPE = "nim"        # "nim" | "openai" — remote endpoints only
# Credentials for a remote LLM endpoint. Both empty is correct for LLM_MODE="local"
# or "none"; with "remote" you need whichever one your endpoint authenticates against.
NVIDIA_API_KEY = ""           # required for build.nvidia.com remote endpoints
OPENAI_API_KEY = ""           # required instead for OpenAI remote endpoints

# ---- GPU device IDs (leave empty to keep the overrides.env defaults: 0 / 1 / 2) ----
RT_CV_DEVICE_ID = ""          # perception (DeepStream)
RT_VLM_DEVICE_ID = ""         # RT-VLM (bp_wh only)
LLM_DEVICE_ID = ""            # LLM NIM (bp_wh + LLM_MODE=local)

# ---- App data (sample videos + perception models) ----
# NGC resource holding the sample videos and perception models.
APP_DATA_RESOURCE = "nvstaging/vss-warehouse/vss-warehouse-app-data:v3.3.0-09152026"

# Where Section 6 downloads and extracts that resource.
# Empty = your home directory, i.e. ~/vss-warehouse-app-data_v<version>/.
DOWNLOAD_DIR = ""

# Reuse app data already extracted on this host instead of downloading it again.
# Empty = download from NGC into DOWNLOAD_DIR. To reuse, point this at the inner
# `vss-warehouse-app-data` directory (the one holding videos/, models/, playback/,
# data_log/) and Section 6 skips the download entirely.
VSS_DATA_DIR_OVERRIDE = ""

# ============================================================
# OPTIONAL overrides
# ============================================================

# Which shipped dataset to run. Only these three are supported — each ships a checked-in
# calibration.json for 2d, 3d and mv3dt, so any dataset pairs with any mode:
#   "nv-warehouse-4cams"                      4 cameras, real footage
#   "warehouse-loading-dock-3cams-synthetic"  3 cameras, synthetic
#   "warehouse-4cams-20mx20m-synthetic"       4 cameras, synthetic
# NUM_STREAMS and DATASET_TYPE are derived from this choice — never set by hand.
# Custom datasets are not supported: they need a calibration run first (see the
# vss-generate-video-calibration skill, or MODE=auto-calibration).
SAMPLE_VIDEO_DATASET = "warehouse-4cams-20mx20m-synthetic"
# "nv-warehouse-4cams" | "warehouse-loading-dock-3cams-synthetic" | "warehouse-4cams-20mx20m-synthetic"

# Use the -sbsa image variants (aarch64 hosts running the SBSA driver, not Tegra).
# False = the default tags. Forced on for HARDWARE_PROFILE="DGX-SPARK", which
# crash-loops on the Tegra build.
USE_SBSA_IMAGES = False

# ============================================================
# DETECTED, not configured
# ============================================================
# These are read off the host and printed by the section that finds them — there is
# nothing to set here:
#   NGC org           Section 3  (from APP_DATA_RESOURCE)
#   Docker storage    Section 4  (root disk vs. a larger mount)
#   HOST_IP           Section 7  (ip route)
#   EXTERNAL_IP       Section 7  (public address, falls back to HOST_IP)
#   Brev secure links Section 7  (BREV_ENV_ID, link domain and prefix)
# If one is detected wrong, export the matching variable before starting Jupyter —
# NGC_CLI_ORG, STORAGE_ROOT, HOST_IP, EXTERNAL_IP — and re-run that section.

In [ ]:
# ---- Validate and resolve configuration ----
import os, shutil

VALID_MODES = ("2d", "3d", "mv3dt", "auto-calibration")
VALID_BP_PROFILES = ("bp_wh", "bp_wh_kafka", "bp_wh_redis", "bp_wh_auto_calib")
# Perception tuning (max_streams_supported, DeepStream tweaks) comes from the
# blueprint-configurator hardware sections; local NIMs additionally need a matching
# services/nim/<slug>/hw-<PROFILE>.env. Both are checked below.
VALID_HARDWARE = ("H100", "L4", "L40S", "RTXA6000", "RTXA6000ADA", "RTXPRO6000BW",
                  "RTXPRO4500BW", "IGX-THOR", "AGX-THOR", "DGX-SPARK", "OTHER")
# Local NIM model -> compose profile slug.
LLM_SLUGS = {
    "nvidia/nemotron-3.5-lightning-30b-a3b": "nemotron-3.5-lightning-30b-a3b",
    "nvidia/NVIDIA-Nemotron-Nano-9B-v2-FP8": "nvidia-nemotron-nano-9b-v2-fp8",
}

assert NGC_CLI_API_KEY, "NGC_CLI_API_KEY is required. Get one at https://ngc.nvidia.com"
assert MODE in VALID_MODES, f"Invalid MODE: {MODE!r}. Must be one of {VALID_MODES}."
assert BP_PROFILE in VALID_BP_PROFILES, (
    f"Invalid BP_PROFILE: {BP_PROFILE!r}. Must be one of {VALID_BP_PROFILES}."
)
# Single auto-calib stack: either selector implies the other.
if MODE == "auto-calibration" or BP_PROFILE == "bp_wh_auto_calib":
    if MODE not in ("auto-calibration",) and BP_PROFILE == "bp_wh_auto_calib":
        print("NOTE: BP_PROFILE=bp_wh_auto_calib forces MODE=auto-calibration "
              "(not 2d/3d/mv3dt).")
        MODE = "auto-calibration"
    if BP_PROFILE != "bp_wh_auto_calib" and MODE == "auto-calibration":
        print("NOTE: MODE=auto-calibration forces BP_PROFILE=bp_wh_auto_calib.")
        BP_PROFILE = "bp_wh_auto_calib"
    if MODE != "auto-calibration" or BP_PROFILE != "bp_wh_auto_calib":
        raise ValueError(
            "Auto-calibration is one stack: MODE=auto-calibration with "
            "BP_PROFILE=bp_wh_auto_calib (COMPOSE_PROFILES=${COMPOSE_PROFILES_WH_AUTO_CALIB})."
        )
if MODE in ("3d", "mv3dt") and BP_PROFILE == "bp_wh":
    raise ValueError(
        f"MODE={MODE} is not compatible with BP_PROFILE=bp_wh. "
        "MODE=3d and MODE=mv3dt support bp_wh_kafka / bp_wh_redis only "
        "(bp_wh requires MODE=2d because the agents stack only runs in 2D). "
        "Camera calibration uses MODE=auto-calibration / BP_PROFILE=bp_wh_auto_calib."
    )
if MODE == "2d" and BP_PROFILE == "bp_wh_auto_calib":
    raise ValueError(
        "BP_PROFILE=bp_wh_auto_calib is not a 2d/3d/mv3dt suffix. "
        "Set MODE=auto-calibration."
    )
assert HARDWARE_PROFILE in VALID_HARDWARE, (
    f"Invalid HARDWARE_PROFILE: {HARDWARE_PROFILE!r}. Must be one of {VALID_HARDWARE}."
)
assert ELASTICSEARCH_MODE in ("cpu", "gpu"), (
    f"Invalid ELASTICSEARCH_MODE: {ELASTICSEARCH_MODE!r}. Must be 'cpu' or 'gpu'."
)

# The agents/NIM stack only exists for MODE=2d + BP_PROFILE=bp_wh; every other variant runs
# with LLM_MODE=none (LLM settings apply only to that combination).
IS_AGENT_PROFILE = (MODE == "2d" and BP_PROFILE == "bp_wh")
if not IS_AGENT_PROFILE:
    LLM_MODE = "none"
assert LLM_MODE in ("local", "remote", "none"), f"Invalid LLM_MODE: {LLM_MODE!r}"
if LLM_MODE == "remote":
    assert REMOTE_LLM_ENDPOINT_URL, (
        "LLM_MODE='remote' requires REMOTE_LLM_ENDPOINT_URL "
        "(endpoint root, no trailing /v1)."
    )
    assert NVIDIA_API_KEY or OPENAI_API_KEY, (
        "LLM_MODE='remote' requires NVIDIA_API_KEY (build.nvidia.com) or OPENAI_API_KEY."
    )
    assert LLM_MODEL_TYPE in ("nim", "openai"), f"Invalid LLM_MODEL_TYPE: {LLM_MODEL_TYPE!r}"

# STREAM_TYPE is derived, never configured: redis only for bp_wh_redis.
STREAM_TYPE = "redis" if BP_PROFILE == "bp_wh_redis" else "kafka"


def _default_dataset(mode, profile):
    if mode == "auto-calibration" or profile == "bp_wh_auto_calib":
        return "warehouse-loading-dock-3cams-synthetic"
    if mode in ("3d", "mv3dt"):
        return "warehouse-4cams-20mx20m-synthetic"
    if profile == "bp_wh":
        return "nv-warehouse-4cams"
    return "warehouse-loading-dock-3cams-synthetic"


# The three shipped datasets: camera count and footage family. Both values are read
# from here rather than from MODE/BP_PROFILE, because dataset and mode are independent
# -- every dataset ships calibration for 2d, 3d and mv3dt. NUM_STREAMS must equal the
# dataset's camera count or perception runs a short stream count with every container
# healthy, and blueprint_config.yml rejects the mismatch at configurator start-up.
SHIPPED_DATASETS = {
    "nv-warehouse-4cams":                     {"num_streams": 4, "dataset_type": "real"},
    "warehouse-loading-dock-3cams-synthetic": {"num_streams": 3, "dataset_type": "synthetic"},
    "warehouse-4cams-20mx20m-synthetic":      {"num_streams": 4, "dataset_type": "synthetic"},
}

DATASET = SAMPLE_VIDEO_DATASET or _default_dataset(MODE, BP_PROFILE)
assert DATASET in SHIPPED_DATASETS, (
    f"Unsupported SAMPLE_VIDEO_DATASET: {DATASET!r}. This notebook deploys the shipped "
    f"datasets only: {sorted(SHIPPED_DATASETS)}. A custom dataset needs a calibration.json "
    "under warehouse-<mode>-app/calibration/sample-data/<name>/ before it can be deployed."
)
NUM_STREAMS = SHIPPED_DATASETS[DATASET]["num_streams"]


# ---- DATASET_TYPE: determined by the dataset, never chosen ----
# Under MODE=3d this selects the whole Sparse4D bundle -- ONNX, engine, anchor and
# labels-<type>.txt -- so it has to track the footage family. Inert for 2d, mv3dt and
# auto-calibration, but always written so the label mount never interpolates to an
# empty path (Docker would create labels-.txt as a directory).
DATASET_TYPE = SHIPPED_DATASETS[DATASET]["dataset_type"]

# ---- COMPOSE_PROFILES selector: the one COMPOSE_PROFILES_WH_* list MODE+BP_PROFILE pick ----
# There is no _MINIMAL branch below, by design: every deployment this notebook makes is the
# full extended stack — ELK, Video Analytics API, HAProxy ingress and monitoring included.
# (A MINIMAL_PROFILE knob used to sit in Section 1 and was silently ignored.) For a minimal
# stack, point COMPOSE_PROFILES at the matching ..._MINIMAL list in generated.env after
# Section 8 and run docker compose yourself; see
# skills/vss-build-vision-ai/references/profiles/warehouse.md, "Profile Service Set".
if BP_PROFILE == "bp_wh_auto_calib" or MODE == "auto-calibration":
    COMPOSE_PROFILES_SELECTOR = "COMPOSE_PROFILES_WH_AUTO_CALIB"
elif BP_PROFILE == "bp_wh":
    COMPOSE_PROFILES_SELECTOR = "COMPOSE_PROFILES_WH_2D"
else:
    _broker = "KAFKA" if BP_PROFILE == "bp_wh_kafka" else "REDIS"
    COMPOSE_PROFILES_SELECTOR = f"COMPOSE_PROFILES_WH_{_broker}_{MODE.upper()}"

# ---- Repo layout: compose lives in-tree under <repo>/deploy/docker ----
DEPLOY_SOURCE_PATH = os.path.expanduser(DEPLOY_SOURCE_PATH)
assert os.path.isdir(DEPLOY_SOURCE_PATH), (
    f"DEPLOY_SOURCE_PATH does not exist: {DEPLOY_SOURCE_PATH!r}. "
    "On Brev the repo is cloned automatically — set DEPLOY_SOURCE_PATH in Section 1."
)
DEPLOYMENTS_DIR   = os.path.join(DEPLOY_SOURCE_PATH, "deploy", "docker")
SCRIPT_DIR        = os.path.join(DEPLOYMENTS_DIR, "scripts")
WAREHOUSE_REL     = "industry-profiles/warehouse-operations"
WAREHOUSE_DIR     = os.path.join(DEPLOYMENTS_DIR, *WAREHOUSE_REL.split("/"))
WAREHOUSE_ENV     = os.path.join(WAREHOUSE_DIR, ".env")
WAREHOUSE_OVERRIDES = os.path.join(WAREHOUSE_DIR, "overrides.env")
WAREHOUSE_GENERATED = os.path.join(WAREHOUSE_DIR, "generated.env")

for _p in (os.path.join(DEPLOYMENTS_DIR, "compose.yml"),
           os.path.join(DEPLOYMENTS_DIR, "containers.env"),
           WAREHOUSE_ENV, WAREHOUSE_OVERRIDES):
    assert os.path.isfile(_p), (
        f"Missing {_p}. DEPLOY_SOURCE_PATH does not look like a "
        "video-search-and-summarization checkout."
    )

# bp_wh (agents) is rejected on these platforms by the blueprint-configurator validation.
if BP_PROFILE == "bp_wh" and HARDWARE_PROFILE in ("IGX-THOR", "DGX-SPARK"):
    raise ValueError(
        f"BP_PROFILE=bp_wh is not supported on {HARDWARE_PROFILE} "
        "(blueprint_config.yml rejects it). Use bp_wh_kafka / bp_wh_redis."
    )

# Warn when the configurator has no tuning section for this hardware: stream counts then
# fall back to NUM_STREAMS with no DeepStream tuning applied.
_bp_config = os.path.join(WAREHOUSE_DIR, "blueprint-configurator", "blueprint_config.yml")
if os.path.isfile(_bp_config):
    with open(_bp_config, encoding="utf-8") as f:
        _tuned = [l.split(":")[0] for l in f if l and l[0].isalnum() and l.rstrip().endswith(":")]
    if HARDWARE_PROFILE not in _tuned:
        print(f"NOTE: blueprint_config.yml has no tuning section for "
              f"HARDWARE_PROFILE={HARDWARE_PROFILE} (tuned: {', '.join(sorted(set(_tuned)))}). "
              "Stream count falls back to NUM_STREAMS with no hardware-specific tuning.")

# A local LLM NIM mounts services/nim/<slug>/hw-<HARDWARE_PROFILE>.env — a missing file
# fails the compose run with an unhelpful "no such file" error, so check it up front.
if LLM_MODE == "local":
    _slug = LLM_SLUGS.get(LLM_NAME)
    assert _slug, (
        f"Unknown LLM_NAME {LLM_NAME!r}. Supported local models: {sorted(LLM_SLUGS)}"
    )
    _hw_env = os.path.join(DEPLOYMENTS_DIR, "services", "nim", _slug,
                           f"hw-{HARDWARE_PROFILE}.env")
    assert os.path.isfile(_hw_env), (
        f"{LLM_NAME} has no sizing file for HARDWARE_PROFILE={HARDWARE_PROFILE} "
        f"({_hw_env} is missing). Pick another model, use HARDWARE_PROFILE=OTHER, "
        "or set LLM_MODE='remote'."
    )

# ---- VSS_DATA_DIR: extracted app-data directory (NOT the repo) ----
if VSS_DATA_DIR_OVERRIDE:
    VSS_DATA_DIR = os.path.expanduser(VSS_DATA_DIR_OVERRIDE)
    APP_DATA_EXTRACT_DIR = os.path.dirname(VSS_DATA_DIR)
else:
    _base = DOWNLOAD_DIR.rstrip("/") if DOWNLOAD_DIR else os.path.expanduser("~")
    _app_data_version = APP_DATA_RESOURCE.split(":", 1)[-1]
    APP_DATA_EXTRACT_DIR = os.path.join(_base, f"vss-warehouse-app-data_v{_app_data_version}")
    VSS_DATA_DIR = os.path.join(APP_DATA_EXTRACT_DIR, "vss-warehouse-app-data")

# ---- Which optional slices this variant deploys (matches COMPOSE_PROFILES_WH_* lists) ----
# Always extended: this notebook never selects a ..._MINIMAL service list.
EXTENDED    = BP_PROFILE in ("bp_wh_kafka", "bp_wh_redis")
HAS_INGRESS = BP_PROFILE in ("bp_wh", "bp_wh_auto_calib") or EXTENDED
HAS_ELK     = BP_PROFILE == "bp_wh" or EXTENDED          # elasticsearch + kibana + logstash
HAS_AGENT   = IS_AGENT_PROFILE                           # vss-agent, vss-agent-ui, va-mcp, RT-VLM
HAS_AUTO_CALIB = MODE == "auto-calibration" or BP_PROFILE == "bp_wh_auto_calib"

# ---- Compose invocation used by Sections 9 / 10 / 13 / 14 ----
COMPOSE_ARGS = [
    "docker", "compose",
    "-f", "compose.yml",
    "-f", "services/infra/compose-no-turn-tcp-relay.yml",
    "--env-file", "containers.env",
    "--env-file", f"{WAREHOUSE_REL}/.env",
    "--env-file", f"{WAREHOUSE_REL}/generated.env",
]


def find_bin(name):
    """Locate a binary, including sbin dirs.

    Jupyter kernels are commonly started with a PATH that omits /usr/sbin and /sbin
    (on this class of host: /…/.venv/bin:/usr/local/bin:/usr/bin:/bin), so a plain
    shutil.which() reports root-only tools such as `ufw` and `sysctl` as missing and
    the caller silently skips work that actually needed doing.
    """
    found = shutil.which(name)
    if found:
        return found
    for d in ("/usr/local/sbin", "/usr/sbin", "/sbin"):
        candidate = os.path.join(d, name)
        if os.path.isfile(candidate) and os.access(candidate, os.X_OK):
            return candidate
    return ""


def env_value(key, default=""):
    """Read KEY from generated.env, falling back to overrides.env then .env."""
    for path in (WAREHOUSE_GENERATED, WAREHOUSE_OVERRIDES, WAREHOUSE_ENV):
        if not os.path.isfile(path):
            continue
        with open(path, encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line.startswith(f"{key}="):
                    continue
                val = line.split("=", 1)[1].split(" #", 1)[0].strip()
                if len(val) > 1 and val[0] == val[-1] and val[0] in "\"'":
                    val = val[1:-1]
                if val and "${" not in val:
                    return val
    return default


COMPOSE_PROJECT = env_value("COMPOSE_PROJECT_NAME", os.environ.get("COMPOSE_PROJECT_NAME", "vss"))


os.environ["NGC_CLI_API_KEY"] = NGC_CLI_API_KEY

print("Configuration:")
print(f"  DEPLOY_SOURCE_PATH:  {DEPLOY_SOURCE_PATH}")
print(f"  deploy/docker:       {DEPLOYMENTS_DIR}")
print(f"  warehouse env files: {WAREHOUSE_REL}/.env + generated.env (from overrides.env)")
print(f"  MODE:                {MODE}")
print(f"  BP_PROFILE:          {BP_PROFILE}")
if HAS_AUTO_CALIB:
    print("  COMPOSE_PROFILES:    ${COMPOSE_PROFILES_WH_AUTO_CALIB}")
    print("  VST routing:         direct (SDRC keys commented in generated.env)")
print(f"  STREAM_TYPE:         {STREAM_TYPE}  (derived)")
print(f"  HARDWARE_PROFILE:    {HARDWARE_PROFILE}")
if BP_PROFILE in ("bp_wh_kafka", "bp_wh_redis"):
    print("  Deployment size:     extended (minimal is not selectable from this notebook)")
print(f"  ELASTICSEARCH_MODE:  {ELASTICSEARCH_MODE}")
print(f"  DATASET:             {DATASET}  ({NUM_STREAMS} streams)")
print(f"  LLM_MODE:            {LLM_MODE}")
if LLM_MODE != "none":
    print(f"  LLM_NAME:            {LLM_NAME}")
if LLM_MODE == "remote":
    print(f"  LLM endpoint:        {REMOTE_LLM_ENDPOINT_URL}  (type: {LLM_MODEL_TYPE})")
print(f"  VSS_DATA_DIR:        {VSS_DATA_DIR}"
      f"{'  (pre-existing, Section 6 skipped)' if VSS_DATA_DIR_OVERRIDE else ''}")
print(f"  NGC key:             {NGC_CLI_API_KEY[:4]}...{NGC_CLI_API_KEY[-4:]}")
print()
print("Service slices for this variant:")
print(f"  HAProxy ingress:     {HAS_INGRESS}")
print(f"  ELK + analytics API: {HAS_ELK}")
print(f"  Agents + Agent UI:   {HAS_AGENT}")
print(f"  Auto-calibration:    {HAS_AUTO_CALIB}")
print()
print("Configuration valid.")

## 2. Prerequisites Check

Verifies the system requirements for the warehouse blueprint:

- **2.1** NVIDIA driver and GPU detection
- **2.2** Docker (tested range [28.3.3, 29.5.0); see Section 4.1), non-root access, cgroupfs driver
- **2.3** NVIDIA Container Toolkit
- **2.4** Linux kernel network settings (IPv6 disabled, socket buffers)
- **2.5** IPv6 localhost entry in `/etc/hosts`
- **2.6** Minimum resources: 10+ CPU cores, 64 GB+ RAM, 500 GB+ disk

If a check fails, install/fix the missing component before proceeding — this cell only reports.

> **2.4 is not applied for you.** `dev-profile.sh` writes `/etc/sysctl.d/99-vss.conf` for the
> developer profiles, but the warehouse path does not, so these need to be set
> by hand if 2.4 warns:
>
> ```bash
> sudo tee /etc/sysctl.d/99-vss.conf >/dev/null <<'EOF'
> net.ipv6.conf.all.disable_ipv6 = 1
> net.ipv6.conf.default.disable_ipv6 = 1
> net.ipv6.conf.lo.disable_ipv6 = 1
> net.core.rmem_max = 5242880
> net.core.wmem_max = 5242880
> EOF
> sudo sysctl --system
> ```

> **GPU count:** `bp_wh` with `LLM_MODE=local` expects three GPUs (perception, RT-VLM, LLM NIM).
> On a smaller host, point `RT_CV_DEVICE_ID` / `RT_VLM_DEVICE_ID` / `LLM_DEVICE_ID` at devices
> that exist — but watch VRAM: RT-VLM alone can claim ~80% of its card, so co-locating it with a
> 9B NIM will not fit on one 48 GB GPU. `LLM_MODE="remote"` (the default) sidesteps this.

In [ ]:
%%bash
# Jupyter kernels often start without /usr/sbin:/sbin on PATH, which makes
# sysctl (in /usr/sbin) look missing and the 2.4 checks report bogus values.
export PATH="$PATH:/usr/local/sbin:/usr/sbin:/sbin"

echo "=== 2.1 GPU & NVIDIA Driver ==="
nvidia-smi --query-gpu=index,name,driver_version,memory.total --format=csv,noheader
echo ""

echo "=== 2.2 Docker ==="
docker --version
docker compose version
if docker ps > /dev/null 2>&1; then
    echo "Docker non-root access: OK"
else
    echo "WARNING: Docker requires sudo."
    echo "  Fix: sudo usermod -aG docker $USER && newgrp docker"
fi
echo ""

echo "=== cgroupfs driver ==="
if docker info 2>/dev/null | grep -qi "cgroupfs"; then
    echo "cgroupfs: OK"
else
    echo "WARNING: cgroupfs driver not configured in /etc/docker/daemon.json"
    echo "  Fix: add exec-opts native.cgroupdriver=cgroupfs, then restart docker"
fi
echo ""

echo "=== 2.3 NVIDIA Container Toolkit ==="
if docker run --rm --gpus all ubuntu:22.04 nvidia-smi > /dev/null 2>&1; then
    echo "NVIDIA Container Toolkit: OK"
else
    echo "WARNING: NVIDIA Container Toolkit not working."
    echo "  Install: https://docs.nvidia.com/datacenter/cloud-native/container-toolkit/latest/install-guide.html"
fi
echo ""

echo "=== 2.4 Linux Kernel Settings ==="
IPV6=$(sysctl -n net.ipv6.conf.all.disable_ipv6 2>/dev/null || echo "unset")
RMEM=$(sysctl -n net.core.rmem_max 2>/dev/null || echo "0")
echo "net.ipv6.conf.all.disable_ipv6 = $IPV6 (need: 1)"
echo "net.core.rmem_max = $RMEM (need: 5242880)"
if [ "$IPV6" != "1" ]; then
    echo "WARNING: IPv6 not disabled. See Section 2 notes for the sysctl fix."
fi
if [ "${RMEM:-0}" -lt 5242880 ] 2>/dev/null; then
    echo "WARNING: rmem_max too low. See Section 2 notes for the sysctl fix."
fi
echo ""

echo "=== 2.5 IPv6 Localhost Entry ==="
HOSTS_ENTRY=$(grep "^::1" /etc/hosts 2>/dev/null || echo "(not found)")
echo "  /etc/hosts ::1 line: $HOSTS_ENTRY"
if echo "$HOSTS_ENTRY" | grep -q "localhost6"; then
    echo "  IPv6 localhost entry: OK"
elif echo "$HOSTS_ENTRY" | grep -q "^::1"; then
    echo "  WARNING: must use localhost6 not localhost"
    echo "    Fix: sudo sed -i 's/^::1 localhost ip6/::1 localhost6 ip6/' /etc/hosts"
else
    echo "  INFO: No ::1 entry found (OK if IPv6 is disabled)"
fi
echo ""

echo "=== 2.6 Minimum System Resources ==="
echo "CPU cores: $(nproc) (need: 10+)"
free -h | awk '/^Mem:/ {print "RAM: " $2 " (need: 64 GB+)"}'
df -h / | tail -1 | awk '{print "Disk free: " $4 " of " $2 " (need: 500 GB+)"}'
echo ""

echo "Prerequisites check complete."

## 3. Install NGC CLI

The NGC CLI is required to download the warehouse app data. This cell installs it if not
already present, then configures it with your API key.

In [ ]:
import subprocess, os, shutil

def run(cmd, **kwargs):
    """Run a shell command, raise on failure with output."""
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, **kwargs)
    if r.returncode != 0:
        raise RuntimeError(f"Command failed: {cmd}\n{r.stderr}\n{r.stdout}")
    return r.stdout.strip()

# Check if NGC CLI is already installed
ngc_path = shutil.which("ngc")
if ngc_path:
    ver = run("ngc --version 2>&1 | head -1")
    print(f"NGC CLI already installed: {ver}")
else:
    import platform
    arch = platform.machine()
    if arch in ("aarch64", "arm64"):
        filename = "ngccli_linux_arm64.zip"
    else:
        filename = "ngccli_linux.zip"

    NGC_CLI_VERSION = "4.13.0"
    url = f"https://api.ngc.nvidia.com/v2/resources/nvidia/ngc-apps/ngc_cli/versions/{NGC_CLI_VERSION}/files/{filename}"

    print(f"Installing NGC CLI {NGC_CLI_VERSION} ...")
    run(f"cd /tmp && wget -q --content-disposition '{url}' -O ngc_cli.zip")
    size = os.path.getsize("/tmp/ngc_cli.zip")
    if size < 1000:
        raise RuntimeError(f"NGC CLI download failed — file is only {size} bytes. Check the version URL.")
    print(f"  Downloaded {size / 1024 / 1024:.1f} MB")

    run("cd /tmp && unzip -o ngc_cli.zip")
    run("sudo cp -r /tmp/ngc-cli/* /usr/local/bin/")
    run("rm -rf /tmp/ngc_cli.zip /tmp/ngc-cli")

    ver = run("ngc --version 2>&1 | head -1")
    print(f"  Installed: {ver}")

# Configure NGC CLI with API key. Prefer explicit NGC_CLI_ORG from Section 1;
# fall back to inferring from APP_DATA_RESOURCE's first path segment for back-compat.
_ngc_org = os.environ.get("NGC_CLI_ORG", "").strip() or APP_DATA_RESOURCE.split("/", 1)[0]
print(f"NGC org: {_ngc_org} (export NGC_CLI_ORG to override)")
print(f"Configuring NGC CLI (org={_ngc_org})...")
ngc_dir = os.path.expanduser("~/.ngc")
os.makedirs(ngc_dir, exist_ok=True)
with open(os.path.join(ngc_dir, "config"), "w") as f:
    f.write(f""";WARNING - This is a machine generated file. Do not edit manually.
;WARNING - To update local config settings, see 'ngc config set -h'.

[CURRENT]
apikey = {NGC_CLI_API_KEY}
format_type = ascii
org = {_ngc_org}
""")

print("NGC CLI configured.")
print(run("ngc config current"))

## 4. Docker & Containerd Storage

Docker images and containerd layers for VSS require **~250GB** (NIM models, DeepStream, ELK, etc.). Most GPU cloud instances ship with a small root disk (200-250GB) that **will run out of space** during deployment.

This cell auto-detects whether your root disk is too small and moves Docker and containerd storage to a larger mount. Docker **volumes** (NIM / RT-VLM model caches, VST postgres state, logstash plugins) are kept on the root disk so they survive an instance stop that wipes the ephemeral NVMe; images and layers are re-pulled automatically on the next deploy. Warehouse runtime state — Elasticsearch indices, Kafka data, VST recordings — is not in Docker volumes at all: it is bind-mounted under `$VSS_DATA_DIR/data_log/`.

**Common NVMe mount points** (auto-detected):
- AWS DLAMI: `/opt/dlami/nvme`
- Brev/Crusoe: `/ephemeral`
- Custom RAID: `/data`

To override auto-detection, export `STORAGE_ROOT=/mnt/data` before starting Jupyter.

In [ ]:
import subprocess, json, os, shutil

# Detected, not configured: export STORAGE_ROOT=/mnt/data before starting Jupyter to
# skip auto-detection and force a specific mount.
STORAGE_ROOT = os.environ.get("STORAGE_ROOT", "").strip()

MIN_ROOT_FREE_GB = 350  # If root has less than this free, move storage

# --- Auto-detect large mount ---

def get_disk_free_gb(path):
    """Return free space in GB for the filesystem containing path."""
    st = os.statvfs(path)
    return (st.f_bavail * st.f_frsize) / (1024 ** 3)

def get_disk_total_gb(path):
    st = os.statvfs(path)
    return (st.f_blocks * st.f_frsize) / (1024 ** 3)

def find_large_mount():
    """Look for a large non-root mount suitable for Docker storage."""
    candidates = ["/opt/dlami/nvme", "/ephemeral", "/data"]
    for path in candidates:
        if os.path.isdir(path) and os.path.ismount(path):
            free = get_disk_free_gb(path)
            if free > 200:
                return path, free
    return None, 0

def find_mount_unit(mount_path):
    """Convert a mount path to a systemd mount unit name (e.g. /opt/dlami/nvme -> opt-dlami-nvme.mount)."""
    # Strip leading slash, replace remaining slashes with dashes
    unit = mount_path.strip("/").replace("/", "-") + ".mount"
    # Verify this unit exists on the system
    r = subprocess.run(["systemctl", "cat", unit], capture_output=True, text=True)
    if r.returncode == 0:
        return unit
    return None

root_free = get_disk_free_gb("/")
root_total = get_disk_total_gb("/")

print(f"Root disk: {root_free:.0f} GB free / {root_total:.0f} GB total")

if STORAGE_ROOT:
    large_mount = STORAGE_ROOT
    mount_free = get_disk_free_gb(STORAGE_ROOT)
    print(f"Using override: {STORAGE_ROOT} ({mount_free:.0f} GB free)")
    need_move = True
else:
    large_mount, mount_free = find_large_mount()
    need_move = root_free < MIN_ROOT_FREE_GB and large_mount is not None

    if large_mount:
        print(f"Large mount:    {large_mount} ({mount_free:.0f} GB free)")
    else:
        print("No large ephemeral mount detected.")

    if root_free >= MIN_ROOT_FREE_GB:
        print(f"\nRoot disk has enough space ({root_free:.0f} GB free). No storage move needed.")
    elif not large_mount:
        print(f"\nWARNING: Root disk only has {root_free:.0f} GB free and no large mount was found.")
        print("Deployment may fail due to disk space. Consider attaching a larger volume.")

if need_move:
    DOCKER_DATA_ROOT = os.path.join(large_mount, "docker")
    CONTAINERD_ROOT = os.path.join(large_mount, "containerd")
    VOLUMES_DIR = "/var/lib/docker/volumes"  # Keep volumes on persistent root disk

    print(f"\nMoving Docker and containerd storage to {large_mount}")
    print(f"  Docker images/layers: {DOCKER_DATA_ROOT}")
    print(f"  Containerd:           {CONTAINERD_ROOT}")
    print(f"  Docker volumes:       {VOLUMES_DIR} (stays on root for persistence)")

    # --- Check what needs changing ---
    daemon_json = "/etc/docker/daemon.json"
    config = {}
    try:
        with open(daemon_json) as f:
            config = json.load(f)
    except (FileNotFoundError, json.JSONDecodeError):
        pass

    need_daemon_json = config.get("data-root") != DOCKER_DATA_ROOT

    subprocess.run(["sudo", "mkdir", "-p", DOCKER_DATA_ROOT], check=True)
    subprocess.run(["sudo", "mkdir", "-p", VOLUMES_DIR], check=True)

    volumes_link = os.path.join(DOCKER_DATA_ROOT, "volumes")
    need_volumes_symlink = not (os.path.islink(volumes_link) and os.readlink(volumes_link) == VOLUMES_DIR)

    containerd_link = "/var/lib/containerd"
    need_containerd = not (os.path.islink(containerd_link) and os.readlink(containerd_link) == CONTAINERD_ROOT)

    # Even if symlinks are correct, ensure NVMe target dirs actually exist
    # (they get wiped when ephemeral NVMe is reset on instance stop/start)
    need_target_dirs = not os.path.isdir(DOCKER_DATA_ROOT) or not os.path.isdir(CONTAINERD_ROOT)
    if need_target_dirs:
        print(f"\n  NVMe target dir(s) missing — recreating...")
        subprocess.run(["sudo", "mkdir", "-p", DOCKER_DATA_ROOT, CONTAINERD_ROOT], check=True)

    if not need_daemon_json and not need_volumes_symlink and not need_containerd:
        print(f"\n  Docker data-root already set to {DOCKER_DATA_ROOT}")
        print(f"  Volumes symlink already correct: {volumes_link} -> {VOLUMES_DIR}")
        print(f"  Containerd already symlinked: {containerd_link} -> {CONTAINERD_ROOT}")

        # Always ensure the boot-time restore service is up to date
        # (handles the case where service exists but is missing mount dependencies)
        _update_restore_service = True
        _need_restart = need_target_dirs  # Restart Docker/containerd if we had to recreate dirs
    else:
        _update_restore_service = True
        _need_restart = True

        # Stop Docker AND docker.socket (socket can reactivate Docker and recreate dirs)
        print("\n  Stopping Docker and containerd for storage reconfiguration...")
        subprocess.run(["sudo", "systemctl", "stop", "docker.socket"], check=False)
        subprocess.run(["sudo", "systemctl", "stop", "docker"], check=True)
        subprocess.run(["sudo", "systemctl", "stop", "containerd"], check=True)

        # --- Docker daemon.json ---
        if need_daemon_json:
            config["data-root"] = DOCKER_DATA_ROOT
            new_config = json.dumps(config, indent=2)
            subprocess.run(
                f"echo '{new_config}' | sudo tee {daemon_json}",
                shell=True, check=True, capture_output=True
            )
            print(f"  Docker data-root set to {DOCKER_DATA_ROOT}")
        else:
            print(f"  Docker data-root already set to {DOCKER_DATA_ROOT}")

        # --- Volumes symlink (use ln -sfn for idempotency) ---
        if need_volumes_symlink:
            # ln -sfn: force, no-dereference (replaces existing dir/symlink atomically)
            subprocess.run(["sudo", "rm", "-rf", volumes_link], check=True)
            subprocess.run(["sudo", "ln", "-sfn", VOLUMES_DIR, volumes_link], check=True)
            print(f"  Created symlink: {volumes_link} -> {VOLUMES_DIR}")
        else:
            print(f"  Volumes symlink already correct: {volumes_link} -> {VOLUMES_DIR}")

        # --- Containerd ---
        if need_containerd:
            subprocess.run(["sudo", "mkdir", "-p", CONTAINERD_ROOT], check=True)
            if os.path.isdir(containerd_link) and not os.path.islink(containerd_link):
                # Move existing containerd data
                subprocess.run(f"sudo mv {containerd_link}/* {CONTAINERD_ROOT}/ 2>/dev/null; true",
                               shell=True, check=False)
                subprocess.run(["sudo", "rm", "-rf", containerd_link], check=True)
                print(f"  Containerd data moved to {CONTAINERD_ROOT}")
            elif os.path.lexists(containerd_link):
                subprocess.run(["sudo", "rm", "-f", containerd_link], check=True)
            subprocess.run(["sudo", "ln", "-sfn", CONTAINERD_ROOT, containerd_link], check=True)
            print(f"  Containerd symlinked: {containerd_link} -> {CONTAINERD_ROOT}")
        else:
            print(f"  Containerd already symlinked: {containerd_link} -> {CONTAINERD_ROOT}")

    # --- Install/update boot-time restore service ---
    # Ephemeral NVMe is wiped on instance stop/start. This systemd service
    # recreates the directories before Docker/containerd start so they don't crash-loop.
    # We use RequiresMountsFor= so the service waits for the NVMe to actually be mounted.
    if _update_restore_service:
        unit_name = "docker-nvme-restore.service"
        unit_path = f"/etc/systemd/system/{unit_name}"

        # Build After= line — include the mount unit if systemd knows about it
        after_targets = "local-fs.target"
        mount_unit = find_mount_unit(large_mount)
        if mount_unit:
            after_targets += f" {mount_unit}"

        unit_content = f"""[Unit]
Description=Restore Docker/containerd dirs on ephemeral NVMe
Before=containerd.service docker.service
After={after_targets}
RequiresMountsFor={large_mount}

[Service]
Type=oneshot
ExecStart=/bin/bash -c 'mkdir -p {DOCKER_DATA_ROOT} {CONTAINERD_ROOT}'

[Install]
WantedBy=multi-user.target
"""
        import tempfile
        with tempfile.NamedTemporaryFile(mode='w', suffix='.service', delete=False) as tmp:
            tmp.write(unit_content)
            tmp_path = tmp.name
        subprocess.run(["sudo", "cp", tmp_path, unit_path], check=True)
        os.unlink(tmp_path)
        subprocess.run(["sudo", "systemctl", "daemon-reload"], check=True)
        subprocess.run(["sudo", "systemctl", "enable", unit_name], check=True, capture_output=True)
        print(f"  Installed {unit_name} (restores NVMe dirs on boot, waits for mount)")

    # --- Restart if needed ---
    if _need_restart:
        print("\n  Starting containerd and Docker...")
        subprocess.run(["sudo", "systemctl", "start", "containerd"], check=True)
        subprocess.run(["sudo", "systemctl", "start", "docker.socket"], check=True)
        subprocess.run(["sudo", "systemctl", "start", "docker"], check=True)

    r = subprocess.run(["docker", "info", "--format", "{{.DockerRootDir}}"],
                       capture_output=True, text=True)
    print(f"\n  Docker data-root: {r.stdout.strip()}")
    target = os.readlink(containerd_link) if os.path.islink(containerd_link) else containerd_link
    print(f"  Containerd root:  {target}")
    print(f"\n  Storage configuration complete.")
else:
    if not STORAGE_ROOT and root_free >= MIN_ROOT_FREE_GB:
        print("Skipping storage move.")

### 4.1 Pin Docker version

Pin Docker CE + plugins + containerd.io to a known-good combination (CE **29.4.3**, buildx **0.33.0**, compose **5.1.3**, containerd **2.2.3**). Some Brev launchables ship newer versions than the Warehouse blueprint is tested against; pin explicitly so compose/buildx incompatibilities don't surface in later sections. `apt-mark hold` prevents unattended-upgrades or later cells from drifting the box back.

The cell first reads the installed Docker Engine version: if it already falls in the tested range **[28.3.3, 29.5.0)** the version downgrade is **skipped** — re-pinning to an exact version the platform's apt repo may not carry (e.g. DGX Spark / DGX-OS on arm64) would fail with *version not found* for no benefit. The packages are still `apt-mark hold`-ed at their current versions so the box can't drift past the tested range mid-run. Safe to re-run.

Runs *after* the section 4 storage relocation so the APT download lands on the relocated volume and the dockerd restart triggered by the downgrade picks up the new data-root.

In [ ]:
%%bash
# Pin Docker CE + plugins + containerd.io to a known-good combination, but
# ONLY when the host's Docker is outside the tested range. Some Brev
# launchables ship a newer Docker than the Warehouse blueprint is tested
# against; pin explicitly so compose/buildx incompatibilities don't surface
# mid-deployment.
#
# When the installed Docker already falls in [28.3.3, 29.5.0) the version
# downgrade is skipped: re-pinning to an exact epoch-versioned package that
# the platform's apt repo may not carry (e.g. DGX Spark / DGX-OS on arm64)
# fails with "version not found" for no benefit. The in-range packages are
# still held so the box can't drift past the tested range mid-notebook.
# Idempotent -- safe to re-run.

set -euo pipefail

# Tested Docker Engine range.
MIN_DOCKER_VERSION="28.3.3"
MAX_DOCKER_VERSION="29.5.0"

# Packages frozen with `apt-mark hold` so unattended-upgrades / later
# `apt-get install` calls can't drift the box before the notebook finishes.
HOLD_PKGS="docker-ce docker-ce-cli docker-buildx-plugin docker-compose-plugin containerd.io"

version_ge() { [ "$(printf '%s\n%s\n' "$2" "$1" | sort -V | head -n1)" = "$2" ]; }
version_lt() { [ "$1" != "$2" ] && [ "$(printf '%s\n%s\n' "$1" "$2" | sort -V | head -n1)" = "$1" ]; }

DOCKER_VERSION="$(docker version --format '{{.Server.Version}}' 2>/dev/null || true)"
if [ -n "$DOCKER_VERSION" ] \
   && version_ge "$DOCKER_VERSION" "$MIN_DOCKER_VERSION" \
   && version_lt "$DOCKER_VERSION" "$MAX_DOCKER_VERSION"; then
  echo "Docker $DOCKER_VERSION is within the tested range [$MIN_DOCKER_VERSION, $MAX_DOCKER_VERSION); skipping the Docker version pin."
  # No downgrade needed, but still hold the in-range packages at their
  # current versions so unattended-upgrades / later apt-get calls can't drift
  # the box past the tested range for the remainder of the notebook.
  sudo apt-mark hold $HOLD_PKGS
  exit 0
fi

if [ -n "$DOCKER_VERSION" ]; then
  echo "Docker $DOCKER_VERSION is outside the tested range [$MIN_DOCKER_VERSION, $MAX_DOCKER_VERSION); pinning to known-good versions."
else
  echo "Could not read the installed Docker version; pinning to known-good versions."
fi

# Read distro info from /etc/os-release (always present on Ubuntu; minimal
# images don't ship `lsb_release`).
. /etc/os-release
DISTRO="${VERSION_ID}"
CODENAME="${UBUNTU_CODENAME:-${VERSION_CODENAME}}"

# Versions hard-coded to what shipped alongside docker-ce 29.4.3 on the
# Docker apt repo (verified against download.docker.com + upstream GitHub
# release timestamps). When bumping DOCKER_CE_VER, bump these four together.
DOCKER_CE_VER="5:29.4.3-1~ubuntu.${DISTRO}~${CODENAME}"
BUILDX_VER="0.33.0-1~ubuntu.${DISTRO}~${CODENAME}"
COMPOSE_VER="5.1.3-1~ubuntu.${DISTRO}~${CODENAME}"
CONTAINERD_VER="2.2.3-1~ubuntu.${DISTRO}~${CODENAME}"

# Refresh the APT cache first -- without this, the specific epoch-versioned
# package may not be in the local index and the install would fail with
# version-not-found before any pinning takes effect.
sudo apt-get update -qq

sudo DEBIAN_FRONTEND=noninteractive apt-get install -y \
  --allow-downgrades \
  -o Dpkg::Options::=--force-confdef \
  -o Dpkg::Options::=--force-confold \
  docker-ce="$DOCKER_CE_VER" \
  docker-ce-cli="$DOCKER_CE_VER" \
  docker-buildx-plugin="$BUILDX_VER" \
  docker-compose-plugin="$COMPOSE_VER" \
  containerd.io="$CONTAINERD_VER"

# Hold so unattended-upgrades / later `apt-get install` calls don't drift
# the box back to newer versions before the rest of the notebook runs.
sudo apt-mark hold $HOLD_PKGS

## 5. Docker Login

Authenticate with the NVIDIA Container Registry (`nvcr.io`) to pull deployment images.

In [ ]:
import subprocess

result = subprocess.run(
    ["docker", "login", "nvcr.io",
     "--username", "$oauthtoken",
     "--password", NGC_CLI_API_KEY],
    capture_output=True, text=True
)
if result.returncode == 0:
    print("Docker login to nvcr.io: OK")
else:
    print(f"Docker login FAILED:\n{result.stderr}")
    raise RuntimeError("Docker login to nvcr.io failed")

## 6. Get the Warehouse App Data

**Compose files are in-tree** — nothing to download for compose. Section 1 already verified
that `DEPLOY_SOURCE_PATH` points at the repo.

App data (sample videos + perception models) comes from NGC:

- Default resource: `nvstaging/vss-warehouse/vss-warehouse-app-data:v3.3.0-09152026`
- Extracted to `<DOWNLOAD_DIR|~>/vss-warehouse-app-data_v<version>/vss-warehouse-app-data`, where `<version>` is the tag from `APP_DATA_RESOURCE` above (`v3.3.0-09152026` by default, extracting to `..._vv3.3.0-09152026`) — Section 1 derives this path, so changing the tag moves it automatically
- That extracted directory is what Section 8 writes as `VSS_DATA_DIR`
  (**not** the repo path — pointing `VSS_DATA_DIR` at the repo makes the configurator fail to
  find the dataset and the stack stalls)

> **First run only** — if the app-data directory already exists on disk the download is skipped.
> Set `VSS_DATA_DIR_OVERRIDE` in Section 1 to reuse an existing extraction.

In [ ]:
import os, subprocess


def run(cmd, cwd=None):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, cwd=cwd)
    if r.returncode != 0:
        raise RuntimeError(f"Command failed: {cmd}\n{r.stderr}\n{r.stdout}")
    return r.stdout.strip()


print(f"Compose bundle (in-tree): {DEPLOYMENTS_DIR}")
print(f"  compose.yml:      {os.path.join(DEPLOYMENTS_DIR, 'compose.yml')}")
print(f"  containers.env:   {os.path.join(DEPLOYMENTS_DIR, 'containers.env')}")
print(f"  warehouse .env:   {WAREHOUSE_ENV}")
print(f"  overrides.env:    {WAREHOUSE_OVERRIDES}")
print()

if VSS_DATA_DIR_OVERRIDE:
    assert os.path.isdir(VSS_DATA_DIR), (
        f"VSS_DATA_DIR_OVERRIDE points at a missing directory: {VSS_DATA_DIR}"
    )
    print(f"Using existing app data: {VSS_DATA_DIR}")
elif os.path.isdir(VSS_DATA_DIR):
    print(f"App data already present: {VSS_DATA_DIR}")
else:
    print(f"Downloading app data ({APP_DATA_RESOURCE}) ...")
    _base_dir = os.path.dirname(APP_DATA_EXTRACT_DIR) or "."
    os.makedirs(_base_dir, exist_ok=True)
    try:
        run(f"ngc registry resource download-version {APP_DATA_RESOURCE!r}", cwd=_base_dir)
    except RuntimeError as exc:
        _org = APP_DATA_RESOURCE.split("/", 1)[0]
        raise RuntimeError(
            f"Could not download {APP_DATA_RESOURCE}.\n"
            f"  - Confirm NGC_CLI_API_KEY can read the '{_org}' org (ngc config current).\n"
            f"  - A not-found error means this version is not published under '{_org}'; "
            f"list what is available with:\n"
            f"      ngc registry resource list '{_org}/vss-warehouse/*'\n"
            f"Underlying error: {exc}"
        ) from exc
    assert os.path.isdir(APP_DATA_EXTRACT_DIR), (
        f"ngc reported success but {APP_DATA_EXTRACT_DIR} does not exist. Check NGC access "
        f"to org {APP_DATA_RESOURCE.split('/', 1)[0]}, and compare against what landed in "
        f"{_base_dir} (ls -d vss-warehouse-app-data_*)."
    )
    tarball = os.path.join(APP_DATA_EXTRACT_DIR, "vss-warehouse-app-data.tar.gz")
    print(f"Extracting {os.path.basename(tarball)} ...")
    run(f"tar -xf {tarball!r}", cwd=APP_DATA_EXTRACT_DIR)
    print("App data extracted.")

run(f"sudo chmod -R 777 {VSS_DATA_DIR!r}")

# Sanity-check the layout compose and the configurator expect.
_videos = os.path.join(VSS_DATA_DIR, "videos", DATASET)
_models = os.path.join(VSS_DATA_DIR, "models")
print()
print(f"VSS_DATA_DIR:     {VSS_DATA_DIR}")
print(f"  models/:        {'OK' if os.path.isdir(_models) else 'MISSING'}")
if os.path.isdir(_videos) and os.listdir(_videos):
    print(f"  videos/{DATASET}: {len(os.listdir(_videos))} file(s)")
else:
    print(f"  videos/{DATASET}: MISSING or EMPTY")
    print("    WARNING: the selected dataset has no videos. Section 9 creates the")
    print("    directory anyway, so the stack will come up with no streams. Check")
    print("    SAMPLE_VIDEO_DATASET / APP_DATA_RESOURCE in Section 1.")
print()
print("Deployment artifacts ready.")

## 7. Detect Network Configuration

Auto-detects the internal (`HOST_IP`) and external (`EXTERNAL_IP`) addresses. On NAT'd cloud
instances (Brev, AWS) these differ — the internal IP is used for container-to-host traffic, the
external one for browser access. Both are printed below. There is nothing to set in Section 1:
if detection is wrong, export `HOST_IP` / `EXTERNAL_IP` before starting Jupyter and re-run.

On Brev, browser traffic goes through a secure link on the HAProxy ingress port (default
`7777`), so the HAProxy `Host` ACL must accept `<port>-<env>.<brev-domain>` — Section 8 writes
those `VSS_PUBLIC_*` values.

In [ ]:
import os, subprocess


def detect_internal_ip():
    """Detect the internal IP via `ip route`."""
    try:
        out = subprocess.run(
            ["bash", "-c",
             "ip route get 1.1.1.1 | awk '/src/ {for (i=1;i<=NF;i++) if ($i==\"src\") print $(i+1)}'"],
            capture_output=True, text=True, timeout=5,
        )
        return out.stdout.strip()
    except Exception:
        return ""


def detect_external_ip():
    """Detect the external IP via a public echo service."""
    for cmd in ["curl -s --max-time 5 ifconfig.me", "curl -s --max-time 5 icanhazip.com"]:
        try:
            out = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=10)
            ip = out.stdout.strip()
            if ip:
                return ip
        except Exception:
            continue
    return ""


def read_etc_environment():
    """Read key=value pairs from /etc/environment (Brev sets BREV_ENV_ID there)."""
    env = {}
    try:
        with open("/etc/environment") as f:
            for line in f:
                line = line.strip()
                if "=" in line and not line.startswith("#"):
                    key, _, value = line.partition("=")
                    env[key.strip()] = value.strip().strip('"')
    except FileNotFoundError:
        pass
    return env


def brev_link_for_port(port):
    """Resolve the secure link Brev publishes for a destination port.

    Returns (fqdn, public_port), or (None, None) when the port has no link.

    Brev publishes an environment context file listing one entry per exposed
    port. Read the fqdn from it -- never build one from a pattern. The domain
    varies per instance (gobrev.dev, brevlab.com, apps.run.brev.nvidia.com, ...)
    and a constructed hostname that happens to be wrong is indistinguishable
    from a missing link: Brev's edge answers `404 route_not_found` for both,
    and the same wrong hostname lands in VSS_PUBLIC_HOST, so HAProxy's
    known_host ACL then rejects the *correct* URL too.

    Key names are matched loosely so a schema tweak degrades to "no link found"
    (which is reported) rather than a wrong hostname (which is not).
    """
    global BREV_CONTEXT_PROBLEM
    path = os.environ.get("BREV_ENVIRONMENT_CONTEXT_PATH", "/etc/brev/environment-context.json")
    BREV_CONTEXT_PROBLEM = None
    try:
        with open(path, encoding="utf-8") as f:
            data = json.load(f)
    except PermissionError:
        # Some Brev images ship this file readable only by root, and Jupyter runs
        # as the instance user. Passwordless sudo is the norm on Brev, so retry
        # through it rather than reporting a link that is actually published.
        proc = subprocess.run(["sudo", "-n", "cat", path], capture_output=True, text=True)
        if proc.returncode != 0:
            BREV_CONTEXT_PROBLEM = (f"{path} is not readable by {os.environ.get('USER', 'this user')} "
                                    "and `sudo -n cat` failed")
            return None, None
        try:
            data = json.loads(proc.stdout)
        except ValueError as exc:
            BREV_CONTEXT_PROBLEM = f"{path} is not valid JSON ({exc})"
            return None, None
    except FileNotFoundError:
        BREV_CONTEXT_PROBLEM = f"{path} does not exist"
        return None, None
    except (OSError, ValueError) as exc:
        BREV_CONTEXT_PROBLEM = f"{path} could not be read ({exc})"
        return None, None

    entries = data.get("ports") if isinstance(data, dict) else data
    if not isinstance(entries, list):
        return None, None

    def _get(entry, *names):
        for name in names:
            if isinstance(entry, dict) and entry.get(name) not in (None, ""):
                return entry[name]
        return None

    for entry in entries:
        dest = _get(entry, "destination_port", "destinationPort", "target_port", "port")
        try:
            if dest is None or int(dest) != int(port):
                continue
        except (TypeError, ValueError):
            continue
        fqdn = _get(entry, "fqdn", "hostname", "host", "url")
        if not fqdn:
            continue
        fqdn = str(fqdn).split("://", 1)[-1].split("/", 1)[0]
        public = _get(entry, "public_port", "publicPort") or 443
        return fqdn, str(public)
    return None, None


# Detected, not configured: export HOST_IP / EXTERNAL_IP before starting Jupyter to
# override what this cell finds.
HOST_IP = os.environ.get("HOST_IP", "").strip() or detect_internal_ip()
EXTERNAL_IP = os.environ.get("EXTERNAL_IP", "").strip() or detect_external_ip() or HOST_IP

print(f"Internal IP (HOST_IP):   {HOST_IP}   (export HOST_IP to override)")
print(f"External IP:             {EXTERNAL_IP}   (export EXTERNAL_IP to override)")

if HOST_IP and EXTERNAL_IP and HOST_IP == EXTERNAL_IP:
    print("\nInternal == External (direct connection, no NAT)")
elif HOST_IP and EXTERNAL_IP:
    print("\nNAT detected — internal and external IPs differ.")
    print("Browser-facing URLs will use EXTERNAL_IP.")

if not HOST_IP:
    print("\nWARNING: Could not detect the internal IP. Export HOST_IP and re-run this cell.")

# ---- Brev secure links ----
_etc_env = read_etc_environment()
BREV_ENV_ID = os.environ.get("BREV_ENV_ID") or _etc_env.get("BREV_ENV_ID", "")
HAPROXY_HOST_PORT = env_value("HAPROXY_HOST_PORT", "7777")
BREV_LINK_DOMAIN = ""
BREV_BASE_URL = ""

BREV_PUBLIC_PORT = "443"

if BREV_ENV_ID:
    os.environ["BREV_ENV_ID"] = BREV_ENV_ID
    BREV_PUBLIC_HOST, _public_port = brev_link_for_port(HAPROXY_HOST_PORT)
    if BREV_PUBLIC_HOST:
        BREV_PUBLIC_PORT = _public_port
        _link_source = "environment-context.json"
    else:
        # Escape hatch: the full hostname, copied from the Brev console.
        # It has to be the whole host, not just the domain -- the secure-link
        # prefix is a NAME the creator chooses, not the port number (Brev's docs:
        # "Specify its name and port"; port 8888 is commonly published as
        # "jupyter-<env>.<domain>"). So "<port>-<env>.<domain>" is a convention,
        # not a rule, and constructing it is how the wrong hostname gets baked in.
        BREV_PUBLIC_HOST = os.environ.get("BREV_PUBLIC_HOST", "").strip()
        if not BREV_PUBLIC_HOST:
            _ctx = os.environ.get("BREV_ENVIRONMENT_CONTEXT_PATH",
                                  "/etc/brev/environment-context.json")
            if BREV_CONTEXT_PROBLEM:
                _why = BREV_CONTEXT_PROBLEM
                _fix = (f"  sudo chmod a+r {_ctx}\n"
                        "and re-run this cell.")
            else:
                _why = f"no entry for destination port {HAPROXY_HOST_PORT} in {_ctx}"
                _fix = ("Create the link in the Brev console: Secure Links -> + HTTP "
                        f"port -> destination port {HAPROXY_HOST_PORT}, then re-run "
                        "this cell.")
            raise RuntimeError(
                f"Could not resolve the Brev secure link for port {HAPROXY_HOST_PORT}: "
                f"{_why}.\n{_fix}\n"
                "Or copy the endpoint from the Secure Links page and export it "
                "verbatim, e.g.\n"
                "  export BREV_PUBLIC_HOST=7777-abcd1234.gobrev.dev\n"
                "Do not assemble the hostname from a pattern: the link name is "
                "user-chosen and the domain varies per instance, and a wrong value "
                "is baked into VSS_PUBLIC_HOST, after which HAProxy rejects the "
                "correct URL too."
            )
        BREV_PUBLIC_PORT = os.environ.get("BREV_PUBLIC_PORT", "443").strip() or "443"
        _link_source = "BREV_PUBLIC_HOST override"
    BREV_LINK_DOMAIN = BREV_PUBLIC_HOST.split(".", 1)[-1]
    os.environ["BREV_LINK_DOMAIN"] = BREV_LINK_DOMAIN
    BREV_BASE_URL = (f"https://{BREV_PUBLIC_HOST}" if BREV_PUBLIC_PORT == "443"
                     else f"https://{BREV_PUBLIC_HOST}:{BREV_PUBLIC_PORT}")
    print("\n=== Brev environment detected ===")
    print(f"  BREV_ENV_ID:        {BREV_ENV_ID}")
    print(f"  Secure link:        {BREV_PUBLIC_HOST}:{BREV_PUBLIC_PORT}  (from {_link_source})")
    if HAS_INGRESS:
        print(f"  Browser traffic routes through the HAProxy ingress: {BREV_BASE_URL}")
    else:
        print("  NOTE: this variant does not deploy the HAProxy ingress "
              "(minimal kafka/redis) — use SSH port-forwarding for direct ports.")
else:
    BREV_ENV_ID = ""
    BREV_PUBLIC_HOST = ""
    print("\nNo Brev environment detected.")

### 7.1 Brev host setup (Brev only)

Three host-level fixes for Brev instances; the cell is a no-op elsewhere.

1. **UFW** — containers on the Docker bridges (`172.17/16`, `172.18/16`) must be allowed to
   reach host-published ports. Without this, `vss-agent-ui` cannot call the agent at
   `<HOST_IP>:8000` and the browser gets `POST /api/chat` → **500**, even though every
   container reports healthy. The cell verifies the rules afterwards instead of trusting the
   exit code.
2. **CDI spec** — the NVIDIA Container Toolkit writes CDI specs to two paths and neither is
   refreshed when the driver changes. A stale spec makes every GPU container fail at start with
   `failed to stat CDI host device` or `failed to fulfil mount request: open
   /usr/lib/.../libnvidia-*.so.<old-version>`. Regenerate both. Related: the specs also mount
   `/run/nvidia-persistenced/socket`, so `nvidia-persistenced` must be running
   (`sudo systemctl start nvidia-persistenced`) or GPU containers fail the same way.
3. **`/etc/hosts`** — maps the Brev secure-link hostnames to `HOST_IP` so requests from the
   *host* (and your own `curl` checks) do not go out to the secure-link edge, which only
   accepts 443. Note this file is the **host's**: containers on the bridge have their own
   `/etc/hosts` and are unaffected.

In [ ]:
import os, shutil, subprocess

if not BREV_ENV_ID:
    print("Not a Brev instance — nothing to do.")
else:
    def sh(cmd):
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
        return r.returncode, (r.stdout or "") + (r.stderr or "")

    # 1. UFW rules for the Docker bridge subnets.
    # Without these, containers on the bridge cannot reach host-published ports, and
    # vss-agent-ui's server-side call to http://<HOST_IP>:8000/chat/stream times out —
    # the browser sees POST /api/chat -> 500.
    _ufw = find_bin("ufw")
    if not _ufw:
        print("ufw: not installed — skipping")
    else:
        rc, out = sh(f"sudo {_ufw} status")
        if "inactive" in out.lower():
            print("ufw: inactive — no rules needed")
        else:
            for subnet in ("172.17.0.0/16", "172.18.0.0/16"):
                rc, out = sh(f"sudo {_ufw} allow from {subnet}")
                print(f"ufw allow from {subnet}: "
                      f"{out.strip().splitlines()[-1] if out.strip() else rc}")
            # Verify rather than trust: a silent failure here costs an hour of debugging.
            rc, out = sh(f"sudo {_ufw} status")
            still_missing = [s for s in ("172.17.", "172.18.") if s not in out]
            if still_missing:
                print(f"  WARNING: ufw is active but no rule matches {still_missing}. "
                      "Containers may not reach host ports; /api/chat will 500.")
            else:
                print("  verified: both docker bridge subnets are allowed")

    # 2. Regenerate the CDI specs in BOTH locations.
    if find_bin("nvidia-ctk"):
        for path in ("/etc/cdi/nvidia.yaml", "/var/run/cdi/nvidia.yaml"):
            rc, out = sh(f"sudo {find_bin('nvidia-ctk')} cdi generate --output={path}")
            print(f"CDI {path}: {'OK' if rc == 0 else 'FAILED'}")
            if rc != 0:
                print(f"  {out.strip()[-300:]}")
    else:
        print("nvidia-ctk: not installed — skipping CDI regeneration")

    # 3. Resolve the Brev secure-link domains locally (idempotent).
    try:
        with open("/etc/hosts", encoding="utf-8") as f:
            hosts = f.read()
    except OSError:
        hosts = ""
    for port in (HAPROXY_HOST_PORT, env_value("VST_INGRESS_HOST_PORT", "30888")):
        # Use the fqdn Brev actually published: the prefix is not always the port
        # number (8888 is published as "jupyter-<env>.<domain>"), so a constructed
        # hostname can map a name that does not exist while the real one still
        # resolves to the public edge.
        domain, _ = brev_link_for_port(port)
        if not domain:
            print(f"/etc/hosts: no secure link published for port {port} — skipping")
            continue
        if domain in hosts:
            print(f"/etc/hosts: {domain} already present")
        else:
            rc, out = sh(f"echo '{HOST_IP} {domain}' | sudo tee -a /etc/hosts")
            print(f"/etc/hosts: added {HOST_IP} {domain}" if rc == 0 else f"/etc/hosts: FAILED {out}")
    print()
    print("Brev host setup complete.")

## 8. Generate the deployment env file

Compose reads three env files, later ones winning:

```text
containers.env                                   # image registry + tags
industry-profiles/warehouse-operations/.env      # profile defaults (checked in)
industry-profiles/warehouse-operations/generated.env   # this deployment (git-ignored)
```

This cell builds `generated.env`: it copies the checked-in `overrides.env`, appends any
`services/vios/compose-defaults.env` key neither file defines, and then writes the values this
deployment owns — `MODE`, `BP_PROFILE`, `HARDWARE_PROFILE`, `STREAM_TYPE`, `SAMPLE_VIDEO_DATASET`,
`NUM_STREAMS`, `DATASET_TYPE`, `COMPOSE_PROFILES`, host paths, IPs, credentials and LLM settings.

`overrides.env` is **not modified** — it is tracked in git, and `generated.env` is not, which is
also why the NGC key belongs in the generated file rather than the checked-in one.

Two details that are easy to miss:

- **`BP_CONFIGURATOR_ENV_FILE`** is pointed at `generated.env`. `bp-configurator-<mode>` does not
  read its environment through Compose interpolation — it loads its `env_file` directly — so
  without this it would read the checked-in `overrides.env` and bake the `<HOST_IP>` placeholder
  into every config it renders.
- **auto-calibration** deploys no `sdr-controller`, so `VST_USE_SDRC`,
  `STREAM_PROCESSOR_MODULE_ENDPOINT` and `VST_NGINX_MODE` are commented out, letting the direct-VST
  defaults in `services/vios/vst.env` apply. A later env layer can override a variable but cannot
  un-set one, so they have to be commented rather than reassigned.


In [ ]:
import os, re, shutil

# ---------------------------------------------------------------
# Build industry-profiles/warehouse-operations/generated.env
# ---------------------------------------------------------------

def set_env_var(lines, key, value):
    """Set KEY=value: replace an active line, else uncomment a commented one, else append."""
    active = re.compile(rf"^{re.escape(key)}=")
    commented = re.compile(rf"^#\s*{re.escape(key)}=")
    for i, line in enumerate(lines):
        if active.match(line):
            lines[i] = f"{key}={value}\n"
            return
    for i, line in enumerate(lines):
        if commented.match(line):
            lines[i] = f"{key}={value}\n"
            return
    if lines and not lines[-1].endswith("\n"):
        lines.append("\n")
    lines.append(f"{key}={value}\n")


def comment_env_var(lines, key):
    """Comment out an active KEY= line (no-op if absent)."""
    active = re.compile(rf"^{re.escape(key)}=")
    for i, line in enumerate(lines):
        if active.match(line):
            lines[i] = f"# {line.lstrip()}"
            return True
    return False


def env_keys(path):
    keys = set()
    if not os.path.isfile(path):
        return keys
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                keys.add(line.split("=", 1)[0].strip())
    return keys


# 1. Start from the checked-in overrides (never modified in place).
shutil.copyfile(WAREHOUSE_OVERRIDES, WAREHOUSE_GENERATED)
with open(WAREHOUSE_GENERATED, encoding="utf-8") as f:
    lines = f.readlines()
if lines and not lines[-1].endswith("\n"):
    lines.append("\n")

# 2. auto-calibration runs direct VST: comment the SDRC trio so vst.env defaults apply.
if HAS_AUTO_CALIB:
    for _key in ("VST_USE_SDRC", "STREAM_PROCESSOR_MODULE_ENDPOINT", "VST_NGINX_MODE"):
        if comment_env_var(lines, _key):
            print(f"  commented {_key} (auto-calibration uses direct VST)")

# 3. Append compose-wide defaults neither .env nor the overrides already define.
_defaults = os.path.join(DEPLOYMENTS_DIR, "services", "vios", "compose-defaults.env")
if os.path.isfile(_defaults):
    _defined = env_keys(WAREHOUSE_ENV) | {
        l.split("=", 1)[0].strip() for l in lines
        if l.strip() and not l.strip().startswith("#") and "=" in l
    }
    with open(_defaults, encoding="utf-8") as f:
        for line in f:
            if not line.strip() or line.lstrip().startswith("#") or "=" not in line:
                continue
            if line.split("=", 1)[0].strip() not in _defined:
                lines.append(line if line.endswith("\n") else line + "\n")

# 4. Values this deployment owns.
updates = {
    "VSS_APPS_DIR": DEPLOYMENTS_DIR,
    "VSS_DATA_DIR": VSS_DATA_DIR,
    "HOST_IP": HOST_IP,
    "NGC_CLI_API_KEY": NGC_CLI_API_KEY,
    "MODE": MODE,
    "BP_PROFILE": BP_PROFILE,
    "ELASTICSEARCH_MODE": ELASTICSEARCH_MODE,
    "HARDWARE_PROFILE": HARDWARE_PROFILE,
    "STREAM_TYPE": STREAM_TYPE,
    "SAMPLE_VIDEO_DATASET": DATASET,
    "NUM_STREAMS": NUM_STREAMS,
    "DATASET_TYPE": DATASET_TYPE,
    "COMPOSE_PROFILES": "${%s}" % COMPOSE_PROFILES_SELECTOR,
    # bp-configurator loads its env_file directly, bypassing --env-file layering.
    "BP_CONFIGURATOR_ENV_FILE": WAREHOUSE_GENERATED,
}
if EXTERNAL_IP:
    updates["EXTERNAL_IP"] = EXTERNAL_IP

# LLM / VLM: only 2d + bp_wh hosts a NIM LLM and the RT-VLM; everything else is headless.
if IS_AGENT_PROFILE:
    updates["LLM_MODE"] = LLM_MODE
    updates["VLM_MODE"] = "none"          # warehouse always uses the integrated RT-VLM
    updates["VLM_NAME_SLUG"] = "none"
    updates["LLM_NAME"] = LLM_NAME
    if LLM_MODE == "remote":
        updates["LLM_NAME_SLUG"] = "none"
        updates["LLM_BASE_URL"] = REMOTE_LLM_ENDPOINT_URL
        updates["LLM_MODEL_TYPE"] = LLM_MODEL_TYPE
    else:
        updates["LLM_NAME_SLUG"] = LLM_SLUGS[LLM_NAME]
    if NVIDIA_API_KEY:
        updates["NVIDIA_API_KEY"] = NVIDIA_API_KEY
    if OPENAI_API_KEY:
        updates["OPENAI_API_KEY"] = OPENAI_API_KEY
else:
    updates.update({"LLM_MODE": "none", "VLM_MODE": "none",
                    "LLM_NAME_SLUG": "none", "VLM_NAME_SLUG": "none"})

# GPU device IDs (only when explicitly set in Section 1).
for _key, _value in (("RT_CV_DEVICE_ID", RT_CV_DEVICE_ID),
                     ("RT_VLM_DEVICE_ID", RT_VLM_DEVICE_ID),
                     ("LLM_DEVICE_ID", LLM_DEVICE_ID)):
    if str(_value).strip() != "":
        updates[_key] = f"'{_value}'"

# Public ingress: Brev terminates TLS on 443 and forwards to the HAProxy port.
if BREV_ENV_ID:
    updates.update({
        "VSS_PUBLIC_HTTP_PROTOCOL": "https",
        "VSS_PUBLIC_WS_PROTOCOL": "wss",
        "VSS_PUBLIC_HOST": BREV_PUBLIC_HOST,
        "VSS_PUBLIC_PORT": BREV_PUBLIC_PORT,
        # coturn must advertise a routable address, not the proxy hostname.
        "TURN_PUBLIC_HOST": "${EXTERNAL_IP}",
    })
else:
    updates.update({
        "VSS_PUBLIC_HTTP_PROTOCOL": "http",
        "VSS_PUBLIC_WS_PROTOCOL": "ws",
        "VSS_PUBLIC_HOST": "${EXTERNAL_IP}",
        "VSS_PUBLIC_PORT": "${HAPROXY_HOST_PORT}",
        "TURN_PUBLIC_HOST": "${VSS_PUBLIC_HOST}",
    })

# SBSA image variants: containers.env applies the suffix during compose interpolation only,
# so a service reading a tag as plain configuration never sees it — the bp-configurator
# validates VSS_RT_CV_TAG out of its env_file and rejects DGX-SPARK without 'sbsa'.
if USE_SBSA_IMAGES or HARDWARE_PROFILE == "DGX-SPARK":
    _base_tag = os.environ.get("VSS_CONTAINER_TAG") or env_value("VSS_CONTAINER_TAG") or "develop-latest"
    for _key in ("VSS_RT_CV_TAG", "VSS_RT_EMBED_TAG", "VSS_RT_VLM_TAG", "VSS_VIDEO_SUMMARIZATION_TAG"):
        updates[_key] = f"{_base_tag}-sbsa"

for _key, _value in updates.items():
    set_env_var(lines, _key, _value)

with open(WAREHOUSE_GENERATED, "w", encoding="utf-8") as f:
    f.writelines(lines)

print(f"Wrote {WAREHOUSE_GENERATED}")
for _key, _value in updates.items():
    _shown = "***" if "API_KEY" in _key else _value
    print(f"  {_key}={_shown}")
print()
print(f"overrides.env left unmodified: {WAREHOUSE_OVERRIDES}")


## 9. Deploy

Brings the stack up with `docker compose` directly, from `deploy/docker`:

```bash
docker compose -f compose.yml \
    -f services/infra/compose-no-turn-tcp-relay.yml \
    --env-file containers.env \
    --env-file industry-profiles/warehouse-operations/.env \
    --env-file industry-profiles/warehouse-operations/generated.env \
    up --detach --force-recreate --build
```

The second compose file is the shared TURN relay overlay every warehouse deployment needs; it
stops the TURN TCP relay publishing a host port.

Image pulls reuse the `nvcr.io` session from **Section 5** — run that first if you jumped
straight here.

**This cell is additive, not a clean slate.** It creates the `$VSS_DATA_DIR/data_log` directories,
logs in to `nvcr.io`, then runs `up`. It does **not** tear the previous stack down first, so named
volumes and their contents survive a re-run:

| Kept across a re-run | Cleared only by Section 14 |
|----------------------|----------------------------|
| Model caches (local LLM weights, `rtvi-ngc-model-cache`, `rtvi-hf-cache`) | all of them — `down -v` |
| `elastic-data`, `kafka-data`, `vios_pg_data`, `logstash-libs` | plus `$VSS_DATA_DIR/data_log/*` |

`--force-recreate` still replaces every container, so config changes from Section 8 take effect.
If you want a genuinely clean slate — stale Elasticsearch indices or Kafka offsets can otherwise
survive into the new run — run Section 14 first, then this cell.

First run is **10–20 min** with `LLM_MODE="remote"` (image pulls + TensorRT engine build), or
**25–40 min** with `LLM_MODE="local"` (adds ~18 GB of NIM weights).

> `--pull always` is deliberately **not** passed. `containers.env` defaults to the moving tag
> `develop-latest`, so a re-deploy reuses whatever was pulled the first time. Set
> `PULL_IMAGES = True` in the cell to force a refresh.

Full output is written to `~/deploy_warehouse.log`.


In [ ]:
import os, re, subprocess, time

LOG_FILE = os.path.expanduser("~/deploy_warehouse.log")

# Set True to re-pull every image even when the tag already exists locally.
PULL_IMAGES = False

# ---- Data directories: compose bind-mounts these; Docker would create any
# ---- missing one as root:root, which then needs root to repair.
for _rel in ("data_log/analytics_cache", "data_log/calibration_toolkit",
             "data_log/elastic/data", "data_log/elastic/logs", "data_log/kafka",
             "data_log/redis/data", "data_log/redis/log",
             "data_log/nvstreamer/vst_data", "data_log/vss_video_analytics_api",
             f"videos/{DATASET}", "playback", "models"):
    os.makedirs(os.path.join(VSS_DATA_DIR, *_rel.split("/")), exist_ok=True)
for _rel in ("data_log", "models"):
    subprocess.run(["chmod", "-R", "777", os.path.join(VSS_DATA_DIR, _rel)],
                   capture_output=True)
print(f"Data directories ready under {VSS_DATA_DIR}")

# Registry login is Section 5's job -- image pulls below reuse that session.

# ---- Images this deployment resolves to ----
_images = subprocess.run(COMPOSE_ARGS + ["config", "--images"],
                         cwd=DEPLOYMENTS_DIR, capture_output=True, text=True)
if _images.returncode == 0:
    _resolved = sorted(set(filter(None, _images.stdout.splitlines())))
    print(f"Resolved {len(_resolved)} image(s); e.g. {_resolved[0] if _resolved else '-'}")
else:
    raise RuntimeError(f"`docker compose config --images` failed:\n{_images.stderr}")

if PULL_IMAGES:
    print("Pulling images (PULL_IMAGES=True) ...")
    subprocess.run(COMPOSE_ARGS + ["pull", "--ignore-buildable"], cwd=DEPLOYMENTS_DIR)

cmd = COMPOSE_ARGS + ["up", "--detach", "--force-recreate", "--build"]

print()
print("Command:", " ".join(cmd))
print("Working dir:", DEPLOYMENTS_DIR)
print(f"Logging to {LOG_FILE} — the status block below refreshes as work happens.")
print()

# --- Output parsing: docker compose pull / build / container progress ---
PULLING_RE = re.compile(r"(?:^|\s)(\S+)\s+Pulling\b")
PULLED_RE = re.compile(r"(?:^|\s)(\S+)\s+Pulled\b")
BUILD_STEP_RE = re.compile(r"#\d+\s+\[([^\]]+)\]\s+(.+)")
IMAGE_BUILT_RE = re.compile(r"(?:^|\s)(\S+)\s+Built\b")
CONTAINER_RE = re.compile(r"^\s*Container\s+(\S+)\s+(.+?)\s*$")
ERROR_RE = re.compile(r"(?:^|\s)(ERROR|error during connect|failed to|Error response from daemon)")

errors = []
images_pulling, images_pulled = set(), set()
builds, builds_done, containers = {}, set(), {}
STATUS_REFRESH_INTERVAL_S = 10
last_refresh = 0.0
_t0 = time.monotonic()


def _elapsed():
    return time.monotonic() - _t0


def print_status():
    print("=" * 60, flush=True)
    print(f"Elapsed {_elapsed():.0f}s  |  errors: {len(errors)}", flush=True)
    print(f"  pull: {len(images_pulling - images_pulled)} in flight, {len(images_pulled)} done"
          f"  |  build: {len(builds)} active, {len(builds_done)} done"
          f"  |  containers seen: {len(containers)}", flush=True)
    if builds:
        tail = list(builds.items())[-3:]
        print("  build:", "; ".join(f"{k}: {v[:48]}" for k, v in tail), flush=True)
    if containers:
        tail = list(containers.items())[-5:]
        print("  last containers:", "; ".join(f"{n}={s}" for n, s in tail), flush=True)
    if errors:
        print(f"  last error: {errors[-1][:200]}", flush=True)
    print("=" * 60, flush=True)


def _maybe_refresh():
    global last_refresh
    now = time.monotonic()
    if now - last_refresh > STATUS_REFRESH_INTERVAL_S:
        last_refresh = now
        print_status()


process = subprocess.Popen(
    cmd, stdin=subprocess.DEVNULL, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, cwd=DEPLOYMENTS_DIR, env={**os.environ, "NO_COLOR": "1"},
)
assert process.stdout is not None

rc = None
try:
    with open(LOG_FILE, "w", encoding="utf-8") as log:
        for line in process.stdout:
            log.write(line)
            log.flush()
            stripped = line.rstrip()

            if ERROR_RE.search(stripped):
                errors.append(stripped)
                print(stripped, flush=True)
                continue

            m = PULLING_RE.search(stripped)
            if m and "Pulled" not in stripped:
                images_pulling.add(m.group(1))
                _maybe_refresh()
                continue

            m = PULLED_RE.search(stripped)
            if m:
                images_pulled.add(m.group(1))
                _maybe_refresh()
                continue

            m = BUILD_STEP_RE.match(stripped)
            if m:
                builds[m.group(1)] = m.group(2)
                _maybe_refresh()
                continue

            m = IMAGE_BUILT_RE.search(stripped)
            if m:
                builds_done.add(m.group(1))
                builds.pop(m.group(1), None)
                _maybe_refresh()
                continue

            m = CONTAINER_RE.match(stripped)
            if m:
                name, status = m.group(1), m.group(2)
                containers[name] = "Exited" if status.startswith("Exited") else status
                _maybe_refresh()

        rc = process.wait()
        log.write(f"\n---- docker compose up finished (exit code {rc}) ----\n")

    print_status()
    if rc == 0:
        print(f"Deployment finished in {_elapsed():.0f}s. Run Section 10 to verify.")
    else:
        print(f"Deployment FAILED (exit code {rc}).")
        if errors:
            print(f"{len(errors)} error line(s) captured — see above.")
        print(f"Full log: {LOG_FILE}")
except KeyboardInterrupt:
    if process.poll() is None:
        process.terminate()
        try:
            process.wait(timeout=120)
        except Exception:
            pass
    raise
finally:
    try:
        process.stdout.close()
    except Exception:
        pass

if rc is None:
    raise RuntimeError(f"docker compose up did not complete normally. See {LOG_FILE}.")
if rc != 0:
    raise RuntimeError(f"docker compose up exited with {rc}. See {LOG_FILE} for full output.")


## 10. Verify Deployment

Re-run this cell every minute or two until everything reports OK. It does four things:

1. **Expected containers** — asks Compose itself which services this deployment selected
   (`docker compose … config`, using the same env files Section 9 deployed with), so the list
   is never out of date with the compose tree. Each container is then inspected:
   - `running` / `healthy` → OK
   - `exited (0)` → OK for one-shot init jobs (`sdrc-*`, `*-init`, `vss-kafka-topics`, …)
   - `restarting`, `exited (≠0)` or missing → reported as a problem
2. **Perception FPS** — DeepStream `PERF` lines from `vss-rtvi-cv` (or `vss-rtvi-cv-mv3dt`).
   The first start compiles the TensorRT engine (cached afterwards under
   `$VSS_DATA_DIR/models/`), so FPS lines can take several minutes to appear.
3. **Host firewall** — with `bp_wh`, the Agent UI reaches the agent through `<HOST_IP>:8000`,
   so an active `ufw` without a rule for the Docker bridges breaks chat (`POST /api/chat` → 500)
   while every container still looks healthy. Checked here rather than left to the browser.
4. **Endpoint checks** — HTTP probes for the services this variant actually deploys.

In [ ]:
import json, os, re, subprocess, urllib.error, urllib.request

# ---------- 1. Expected containers, straight from Compose ----------
_cfg = subprocess.run(COMPOSE_ARGS + ["config", "--format", "json"],
                      cwd=DEPLOYMENTS_DIR, capture_output=True, text=True)
expected = {}
if _cfg.returncode == 0:
    try:
        _services = json.loads(_cfg.stdout).get("services", {})
        for svc, spec in _services.items():
            name = spec.get("container_name")
            if name:
                expected[name] = svc
    except json.JSONDecodeError:
        pass
if not expected:
    print("WARNING: could not resolve the expected service list from Compose — "
          "generated.env is missing or invalid. Has Section 9 run?")
    print(_cfg.stderr.strip()[-500:])

# generated.env is written by Section 8; warn when it disagrees with Section 1
# (e.g. the notebook config changed but Section 9 has not been re-run).
_deployed_mode = env_value("MODE", "")
_deployed_profile = env_value("BP_PROFILE", "")
if expected and (_deployed_mode != MODE or _deployed_profile != BP_PROFILE):
    print(f"WARNING: generated.env describes MODE={_deployed_mode} / "
          f"BP_PROFILE={_deployed_profile}, but Section 1 selects MODE={MODE} / "
          f"BP_PROFILE={BP_PROFILE}. The list below reflects what was deployed. "
          "Re-run Section 9 to deploy the new selection.")

print(f"=== Containers ({len(expected)} expected for "
      f"MODE={_deployed_mode or MODE} / BP_PROFILE={_deployed_profile or BP_PROFILE}) ===")

_ONESHOT_HINTS = ("-init", "init-", "wait-", "topics", "import-", "render-config",
                  "wdm-env-from-config", "health-check", "sensor-bp-wait")


def inspect(name):
    r = subprocess.run(["docker", "inspect", name], capture_output=True, text=True)
    if r.returncode != 0:
        return None
    try:
        return json.loads(r.stdout)[0]
    except (json.JSONDecodeError, IndexError):
        return None


ok, pending, failed = [], [], []
for name in sorted(expected):
    info = inspect(name)
    if info is None:
        (pending if any(h in name for h in _ONESHOT_HINTS) else failed).append((name, "not created"))
        continue
    state = info.get("State", {})
    status = state.get("Status", "?")
    health = (state.get("Health") or {}).get("Status")
    exit_code = state.get("ExitCode", 0)
    label = f"{status}{f' ({health})' if health else ''}"
    if status == "running" and health in (None, "healthy"):
        ok.append((name, label))
    elif status == "running" and health == "starting":
        pending.append((name, label))
    elif status == "exited" and exit_code == 0:
        ok.append((name, "exited (0) — one-shot"))
    else:
        failed.append((name, f"{label} exit={exit_code} restarts={state.get('RestartCount', 0)}"))

for name, label in ok:
    print(f"  OK      {name:<42s} {label}")
for name, label in pending:
    print(f"  WAIT    {name:<42s} {label}")
for name, label in failed:
    print(f"  PROBLEM {name:<42s} {label}")
print(f"\n  {len(ok)} ok, {len(pending)} starting, {len(failed)} problem(s)")
if expected and not ok and not any(inspect(n) for n in list(expected)[:5]):
    print("  Nothing is running — the stack is not deployed (or was torn down). Run Section 9.")
elif failed:
    print("  Inspect a failing container with: docker logs --tail 100 <name>")

# ---------- 2. Perception FPS ----------
if HAS_AUTO_CALIB:
    print("\n=== FPS check skipped (bp_wh_auto_calib deploys no perception container) ===")
else:
    perception = "vss-rtvi-cv-mv3dt" if MODE == "mv3dt" else "vss-rtvi-cv"
    print(f"\n=== FPS check ({perception}) ===")
    r = subprocess.run(["docker", "logs", "--tail", "80", perception],
                       capture_output=True, text=True)
    output = (r.stdout or "") + (r.stderr or "")
    if r.returncode != 0:
        print(f"  docker logs failed for {perception} (exit {r.returncode}). Is the stack up?")
        if output.strip():
            print(output.strip()[-500:])
    else:
        # DeepStream's PERF *header* is the line that contains "FPS":
        #     **PERF:  FPS 0 (Avg)  FPS 1 (Avg)  ...
        # and it is printed once, before any throughput exists. The rows that carry
        # real numbers contain no "FPS" at all:
        #     **PERF:  14.80 (15.46)  14.80 (15.46)  ...
        # So matching "fps" reports success while the TensorRT engine is still
        # building -- match a decimal rate instead, plus the two unambiguous markers.
        _RATE_RE = re.compile(r"\d+\.\d+")

        def _is_throughput_line(line):
            lo = line.lower()
            if "active sources" in lo or ("source_id" in lo and "stream_name" in lo):
                return True
            return "perf" in lo and "fps" not in lo and bool(_RATE_RE.search(lo))

        perf = [l for l in output.splitlines() if _is_throughput_line(l)]
        if perf:
            for line in perf[-8:]:
                print(" ", line)
        else:
            print("  No FPS / PERF output yet — the TensorRT engine is still building on a")
            print("  first run. Re-run this cell in a minute. Last log lines:")
            for line in output.strip().splitlines()[-8:]:
                print("   ", line)

# ---------- 3. Endpoint checks ----------
_ports = {
    "haproxy":  env_value("HAPROXY_HOST_PORT", "7777"),
    "vst":      env_value("VST_INGRESS_HOST_PORT", "30888"),
    "agent":    env_value("VSS_AGENT_HOST_PORT", "8000"),
    "ui":       env_value("VSS_UI_HOST_PORT", "3000"),
    "es":       env_value("ELASTICSEARCH_HOST_PORT", "9200"),
    "kibana":   env_value("KIBANA_HOST_PORT", "5601"),
    "video_analytics": env_value("VIDEO_ANALYTICS_API_HOST_PORT", "8081"),
    "nvstream": env_value("NVSTREAMER_HTTP_HOST_PORT", "31000"),
    "calib_ui": env_value("VSS_AUTO_CALIBRATION_UI_HOST_PORT", "5000"),
}

checks = [
    ("VST",         f"http://localhost:{_ports['vst']}/vst/api/v1/sensor/list"),
    ("NvStreamer",  f"http://localhost:{_ports['nvstream']}/"),
]
if HAS_INGRESS:
    checks.append(("HAProxy ingress", f"http://localhost:{_ports['haproxy']}/"))
if HAS_AGENT:
    checks += [("Agent", f"http://localhost:{_ports['agent']}/health"),
               ("Agent UI", f"http://localhost:{_ports['ui']}/")]
if HAS_ELK:
    checks += [("Elasticsearch", f"http://localhost:{_ports['es']}/"),
               ("Kibana", f"http://localhost:{_ports['kibana']}/kibana/api/status"),
               ("Video Analytics API", f"http://localhost:{_ports['video_analytics']}/livez")]
if HAS_AUTO_CALIB:
    checks.append(("Auto-calibration UI", f"http://localhost:{_ports['calib_ui']}/"))

# A host firewall that blocks the docker bridges breaks the UI -> agent hop:
# vss-agent-ui runs in bridge mode and calls http://<HOST_IP>:<agent port>/chat/stream
# server-side, so the browser gets POST /api/chat -> 500 while every container looks
# healthy. Check the rule rather than waiting for the symptom.
_ufw = find_bin("ufw")
if _ufw and HAS_AGENT:
    _st = subprocess.run(["sudo", _ufw, "status"], capture_output=True, text=True).stdout
    if _st.strip().lower().startswith("status: active"):
        _missing = [s for s in ("172.17.", "172.18.") if s not in _st]
        print("\n=== Host firewall (bridge -> host) ===")
        if _missing:
            print(f"  PROBLEM ufw is active with no rule for {_missing}")
            print("          The Agent UI cannot reach the agent; /api/chat will return 500.")
            print("          Fix: sudo ufw allow from 172.17.0.0/16 && "
                  "sudo ufw allow from 172.18.0.0/16   (or re-run Section 7.1)")
        else:
            print("  OK      docker bridge subnets allowed")

print("\n=== Endpoint checks ===")
for name, url in checks:
    try:
        with urllib.request.urlopen(url, timeout=5) as resp:
            print(f"  OK      {name:<22s} {resp.getcode()}  {url}")
    except urllib.error.HTTPError as e:
        # HAProxy answers 503 on / when the variant has no UI backend — that still
        # proves the ingress is up and routing.
        verdict = "OK" if (name == "HAProxy ingress" and e.code in (404, 503)) else "WARN"
        print(f"  {verdict:<7s} {name:<22s} HTTP {e.code}  {url}")
    except Exception as e:
        print(f"  WAIT    {name:<22s} {type(e).__name__}  {url}")

## 11. Access the UI

Browser-facing traffic goes through the **HAProxy ingress** (host port `7777` by default) when
the variant deploys it — that is `bp_wh`, `bp_wh_auto_calib`, and extended
`bp_wh_kafka`/`bp_wh_redis` — which is every deployment this notebook makes, since it cannot
select a `_MINIMAL` service list. (A hand-deployed minimal stack has no ingress; there you
would use the direct ports or SSH port-forwarding.)

The cell prints the three UIs you actually open in a browser:

| Path | Backend | Available when |
|------|---------|----------------|
| `/` | Agent UI (chat, alerts, dashboards) | `bp_wh` only — other variants return 503, there is no UI backend |
| `/vst/` | VST — streams, recordings | any ingress-enabled variant |
| `/kibana/` | Kibana — behavior-analytics events | `bp_wh` or extended kafka/redis |

The rest of the ingress routes are service APIs, useful for debugging rather than
browsing: `/api/`, `/chat`, `/websocket`, `/static/` (`vss-agent`),
`/video-analytics-api/`, `/behavior-analytics/`, `/alert-bridge/`, `/va-mcp/`. They live on
the same origin, so `curl <base>/video-analytics-api/livez` works without extra setup.

**On Brev:** create **one** secure link — for the ingress port (`7777`). It resolves to
`7777-<env>.<brev-domain>`, which Section 8 already wrote into `VSS_PUBLIC_HOST` (HAProxy 404s
any `Host` it does not know).

**On other clouds:** open the ingress port in your firewall / security group, or forward it over
SSH.

### Streaming limitation

VST **live and recorded video playback does not render through a Brev secure link**. The VST UI
loads and stream lists/recordings are browsable.

In [ ]:
_ports = {
    "haproxy":  env_value("HAPROXY_HOST_PORT", "7777"),
    "vst":      env_value("VST_INGRESS_HOST_PORT", "30888"),
    "kibana":   env_value("KIBANA_HOST_PORT", "5601"),
    "calib_ui": env_value("VSS_AUTO_CALIBRATION_UI_HOST_PORT", "5000"),
}
host = EXTERNAL_IP or HOST_IP

# Every profile this notebook can select ships the HAProxy ingress (bp_wh and
# bp_wh_auto_calib carry it directly; bp_wh_kafka/bp_wh_redis get it from the extended
# list), so there is no no-ingress branch here -- only Brev vs. direct addressing.
if BREV_ENV_ID:
    base = BREV_BASE_URL
    print(f"Access — Brev secure link on port {_ports['haproxy']} ({base}):")
else:
    base = f"http://{host}:{_ports['haproxy']}"
    print("Access — HAProxy ingress:")

if HAS_AGENT:
    print(f"  Agent UI:   {base}/")
print(f"  VST UI:     {base}/vst/")
if HAS_ELK:
    print(f"  Kibana:     {base}/kibana/")

if HAS_AUTO_CALIB:
    print(f"  Auto-calibration UI: http://{host}:{_ports['calib_ui']}/   (direct port, no ingress route)")

if not HAS_AGENT:
    print(f"  (no Agent UI in {BP_PROFILE} — the ingress root returns 503)")

print()
print("If a URL is not reachable from your browser, forward the ingress port over SSH:")
_fwd = _ports["haproxy"]
print(f"  ssh -L {_fwd}:localhost:{_fwd} <user>@{host}       # or use VSCode Remote-SSH")
print()
print("Note: VST live/recorded playback does not render through a Brev secure link.")
print("      The UI loads and streams/recordings are listed.")

## 12. Next Steps

- **`bp_wh`** — open the Agent UI at the ingress root and ask questions about the warehouse
  streams; alerts and dashboards are in the sidebar tabs.
- **Kibana** (`bp_wh` or extended kafka/redis) — the **Discover** and **Dashboard** tabs show
  behavior-analytics events (ROI crossings, proximity alerts, tracking metadata) flowing from
  the perception pipeline. 3D/MV3DT BEV frames land in the `mdx-bev` index, which exists only
  when Elasticsearch is deployed (so not in minimal mode).
- **`bp_wh_auto_calib`** (`MODE=auto-calibration`) — use the auto-calibration UI to calibrate
  cameras, then redeploy with a perception profile. The stack is the single
  `COMPOSE_PROFILES_WH_AUTO_CALIB` list (direct VST; SDRC commented in `generated.env`).

Docs: [VSS Warehouse Blueprint](https://docs.nvidia.com/vss/latest/warehouse-docs/warehouse-toc.html)
and `deploy/docker/README.md` in this repo.

## 13. Stop the Deployment

**Guarded:** set `CONFIRM_STOP = True` in the cell to run it (so "Run All" cannot stop a stack you just deployed).

Stops the containers **without removing them, their volumes, or `$VSS_DATA_DIR`**. Start them
again with `docker compose -p <COMPOSE_PROJECT_NAME> start` — that keeps the volumes and the
data. Re-running Section 9 instead recreates the containers (`--force-recreate`) but keeps the
volumes; Section 14 is the one that removes them.

All warehouse services run under the configured Compose project (`COMPOSE_PROJECT_NAME` in the
warehouse generated environment, default `vss`), so no env files are needed to address them.

In [ ]:
import subprocess

# Guard: this cell is below the deploy, so an unattended "Run All" would otherwise
# stop the stack you just brought up. Flip to True to actually stop.
CONFIRM_STOP = False

if not CONFIRM_STOP:
    raise SystemExit("Section 13 skipped — set CONFIRM_STOP = True to stop the stack.")

print(f"Stopping the warehouse stack (compose project '{COMPOSE_PROJECT}') ...")
r = subprocess.run(["docker", "compose", "-p", COMPOSE_PROJECT, "stop"],
                   capture_output=True, text=True)
print(r.stdout or "")
if r.returncode != 0:
    print(r.stderr)
    raise RuntimeError("docker compose stop failed")
print(f"Stack stopped. Restart with: docker compose -p {COMPOSE_PROJECT} start")

## 14. Teardown

**Guarded:** set `CONFIRM_TEARDOWN = True` in the cell to run it.

Full teardown, in three steps:

```bash
# 1. containers, project network, and ALL named volumes (incl. model caches)
docker compose -p <COMPOSE_PROJECT_NAME> -f compose.yml \
    -f services/infra/compose-no-turn-tcp-relay.yml \
    --env-file containers.env \
    --env-file industry-profiles/warehouse-operations/.env \
    --env-file industry-profiles/warehouse-operations/generated.env \
    down -v --remove-orphans

# 2. bind-mounted runtime state that survives `down -v`
bash scripts/cleanup_all_datalog.sh -e industry-profiles/warehouse-operations/.env

# 3. the per-deployment env file
rm industry-profiles/warehouse-operations/generated.env
```

`-v` is what removes the named volumes; without it Elasticsearch indices, Kafka offsets, Postgres
state **and** the multi-GB model caches all survive. `--remove-orphans` frees the project network
so it can be deleted too.

Step 2 is not optional: `$VSS_DATA_DIR/data_log/` (VST recordings, kafka, elastic, redis,
analytics cache) is bind-mounted from the host, so no `docker` command touches it, and stale
contents poison the next run. The script also reverts the blueprint-configurator `*.backup_*`
files it finds. It needs **root** — if `sudo` prompts for a password the cell prints the command
for you to run yourself rather than hanging on a prompt.

The downloaded app data (videos + models, including the cached TensorRT engine) is **kept** —
only the runtime `data_log` state is cleared. Re-run Sections 8–10 to redeploy.


In [ ]:
import os, subprocess

# Guard: destructive (containers, volumes incl. model caches, and $VSS_DATA_DIR/data_log/*).
# Flip to True to actually tear down.
CONFIRM_TEARDOWN = False

if not CONFIRM_TEARDOWN:
    raise SystemExit("Section 14 skipped — set CONFIRM_TEARDOWN = True to tear the stack down.")


def run(cmd, **kw):
    print("$", " ".join(cmd), flush=True)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1, cwd=DEPLOYMENTS_DIR, **kw)
    for line in p.stdout:
        print("   ", line.rstrip(), flush=True)
    return p.wait()


# ---- 1. Containers, project network, and every named volume ----
# -p targets this project explicitly: COMPOSE_PROJECT_NAME is settable, and a
# plain `down` would leave the volumes and network behind.
rc = run(["docker", "compose", "-p", COMPOSE_PROJECT] + COMPOSE_ARGS[2:]
         + ["down", "-v", "--remove-orphans"])
print(f"compose down exited {rc}\n")

# ---- 2. Bind-mounted data_log + configurator backups (needs root) ----
# `down -v` removes Docker volumes only; data_log lives on the host filesystem.
cleanup = os.path.join(SCRIPT_DIR, "cleanup_all_datalog.sh")
cleanup_cmd = ["sudo", "env", f"VSS_DATA_DIR={VSS_DATA_DIR}", f"VSS_APPS_DIR={DEPLOYMENTS_DIR}",
               "bash", cleanup, "--env-file", WAREHOUSE_ENV]
if subprocess.run(["sudo", "-n", "true"], capture_output=True).returncode == 0:
    rc = run(cleanup_cmd)
    print(f"data_log cleanup exited {rc}\n")
else:
    print("sudo needs a password — run this once yourself, then re-run this cell:\n")
    print("   ", " ".join(cleanup_cmd), "\n")

# ---- 3. The per-deployment env file ----
if os.path.isfile(WAREHOUSE_GENERATED):
    os.remove(WAREHOUSE_GENERATED)
    print(f"Removed {WAREHOUSE_GENERATED}")

print()
print("Teardown complete. App data under $VSS_DATA_DIR (videos, models) is kept.")


In [ ]:
# Optional: remove the downloaded app-data tarball + extracted directory from disk.
# The repo at DEPLOY_SOURCE_PATH is a git checkout and is never removed by this notebook.
#
# import shutil
# shutil.rmtree(APP_DATA_EXTRACT_DIR)   # <home>/vss-warehouse-app-data_v<version>/
# print("App-data artifacts removed.")